# CS-781 Phase 6 (Abstractive Summarization) - `p6_decode_gdn`

**Phase-1 model:** GDN encoder-decoder (144.3M params): the encoder reads the document, the decoder writes the summary.  **This notebook:** Round 3: decoding study on the GDN encoder-decoder (LoRMA, full FT, Houlsby weights): no-repeat 3, sampling, repetition penalty, chunk-wise summaries, shorter output.

The Phase-1 language model is fine-tuned to continue `[document] SUMMARY OF THE JUDGMENT: [summary]` (decoder-only and LSTM) or to decode the summary from the encoded document (GDN encoder-decoder); the loss is computed on the summary tokens only. Long judgments are cut to a token budget (default: head + tail of the document; other strategies in the ablation notebook). N_DEV = 300 documents of train.json are held out as the dev set (fixed for every notebook); epochs are chosen on their teacher-forced loss, and the first DEV_GEN of them are summarised and scored (BERTScore with roberta-large, ROUGE-1/2/L, length, repetition, novelty, numbers). Every finished setup is cached in the bucket; the test set is summarised and zipped (`prediction.json`, list format).

**Kaggle:** GPU T4, Internet on, secrets `HF` and `HF_WRITE`. `QUICK_TEST = True` in cell 1 = a few-minute check. Hyper-parameters are placeholders proposed by the assistant.

## 1. Settings

In [1]:
# [P6-PRELUDE:settings]
import json
# =====================================================================================================
# SETTINGS -- every value is a placeholder you can change. Hyper-parameter values were proposed by the assistant and are NOT tuned.
# =====================================================================================================
SWEEP_NAME = "p6_decode_gdn"
MODEL_KEY = "gdn"
KIND = "gdn"                 # decoder (GQA / MoE) | mor | lstm | gdn
PHASE1_CHECKPOINT = "phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_gdnencdec160Mw_1ep_checkpoint.pt"
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"
PEFT_TARGETS = [".q_proj", ".k_proj", ".v_proj", ".o_proj", ".out_proj"]
IA3_OUT = [".k_proj", ".v_proj"]
IA3_IN = ["ff.2"]

ATTENTION = "causal"
BERTSCORE_MODEL = "roberta-large"     # as in Phase 1 (bert-score's default model for English)

FULL_RUN = True               # True = all training documents (minus the held-out dev split); False = the first MAX_TRAIN_DOCS of a fixed shuffle
MAX_TRAIN_DOCS = 0
N_DEV = 300                     # documents held out from train.json as the dev set (no official dev split): epoch choice on all of them (teacher-forced loss)
DEV_GEN = 100              # dev documents that are also SUMMARISED and scored (BERTScore / ROUGE) after training (generation is the slow part)
MAX_DOC_TOKENS = 32768          # a longer document keeps its first and last halves before any selection

BASE = dict(
    TECHNIQUE="lorma", RANK=16, ALPHA=32, LORA_DROPOUT=0.0, PEFT_TARGETS=PEFT_TARGETS, LORMA_VARIANT="plus",
    ADALORA_INIT_RANK=24, ADALORA_TARGET_RANK=8, ADALORA_DELTA_T=10, ADALORA_ORTH=0.1,
    IA3_OUT=IA3_OUT, IA3_IN=IA3_IN, ADAPTER_KIND="houlsby", ADAPTER_R=64, PARTIAL_LAYERS=2, PROMPT_LEN=20, PREFIX_LEN=16,
    # sequence budget: prompt (selected document + separator) + summary must fit CONTEXT tokens (GDN: encoder <= ENC_MAX_TOK, decoder <= SUM_MAX + 1)
    CONTEXT=2048, SUM_MAX=768, DOC_BUDGET=None,           # SUM_MAX 768 tokens covers 58% of the summaries whole (median 682); DOC_BUDGET None = what is left of CONTEXT
    SELECT="head_tail", HEAD_FRAC=0.7, IMP_HEAD_KEEP=128, ENC_MAX_TOK=1024, SEP_TEXT="SUMMARY OF THE JUDGMENT:",
    # training recipe (an "epoch" = EPOCH_DOCS documents drawn from the training set)
    OPTIMIZER="adamw", ADAMW_LR_WITH_MUON=3e-4,      # adamw | adam | sgd (Nesterov momentum 0.9) | rmsprop | adafactor | lion | muon (Muon on 2-D matrices + AdamW on the rest)
    BATCH=1, ACCUM=16, EVAL_BATCH=4, EPOCHS=3, EPOCH_DOCS=0, LR=3e-4, WEIGHT_DECAY=0.01, WARMUP_FRAC=0.05, GRAD_CLIP=1.0, AMP="fp16", SEED=0,
    TIME_BUDGET_MIN=None, CKPT=False, N_TRAIN=None,      # TIME_BUDGET_MIN: training stops (even inside an epoch) after this many minutes; N_TRAIN: train on the first N training documents only
    # sequence-level knowledge distillation: train on the TEACHER's generated summaries (file made by p6_kd_teacher)
    KD_SUMMARIES=None, KD_MIX=1.0,
    # word-level (logit) KD: the teacher's top-k log-probabilities at every reference-summary position (file made by a KD_TOPK_JOBS teacher job)
    KD_TOPK=None, KD_ALPHA=0.5, KD_TEMPERATURE=2.0,
    DAPT_EPOCHS=0, DAPT_LR=None,          # domain-adaptive pre-training on the training judgments before fine-tuning (0 = off; LR None = the technique's LR)
    SECTION_TAIL_FRAC=0.2,                # SELECT="sections": share of the budget kept from the end of the judgment        # KD_MIX = share of epochs' examples that use the teacher summary (1.0 = always, when the document has one)
    SAVE_WEIGHTS=True)
LR_BY_TECH = dict(full_ft=3e-5, partial=1e-4, bitfit=1e-3, adapters=5e-4, prompt=3e-3, prefix=3e-3, ia3=3e-3, frozen=3e-4)

# decoding (generation) settings used for the dev scores and the submission
GEN = dict(STRATEGY="greedy", NO_REPEAT_NGRAM=4, REP_PENALTY=1.0, MIN_NEW=None, MAX_NEW=None, TEMPERATURE=0.7, TOP_P=0.9, GEN_BATCH=8, USE_CACHE=True)   # None = from the data

SWEEP = [   # (name, settings that differ from BASE). Order = run order.

]
GEN.update({})
GEN_OVERRIDES = {}      # setup name -> dict of GEN changes (optional)
KD_TOPK_JOBS = []    # [(trained weights p6w_*.pt, output file, k)]: the teacher's top-k log-probs on the reference summaries of all training documents
KD_JOBS = []         # [(trained weights p6w_*.pt, output file, number of training documents)]: the teacher summarises training documents for KD

TEST_GEN = True             # summarise the test set (submission files) after each full-data setup
KD_GEN_BATCH = 16             # generation batch for the teacher job
KD_WAIT_MIN = 240             # a KD student waits up to this long for the teacher summaries to appear in the bucket
# decoding study on saved weights (decode notebooks only): (job name, weights file p6w_*.pt, GEN changes, input-selection changes, chunks (0 = one prompt))
DECODE = [
    ('lorma_norep3', 'p6w_p6_gdn__gdn_lorma.pt', {'NO_REPEAT_NGRAM': 3}, {}, 0),
    ('lorma_sample_t0.7_p0.9', 'p6w_p6_gdn__gdn_lorma.pt', {'STRATEGY': 'sample', 'TEMPERATURE': 0.7, 'TOP_P': 0.9}, {}, 0),
    ('lorma_norep3_sample', 'p6w_p6_gdn__gdn_lorma.pt', {'NO_REPEAT_NGRAM': 3, 'STRATEGY': 'sample', 'TEMPERATURE': 0.7, 'TOP_P': 0.9}, {}, 0),
    ('lorma_reppen1.2', 'p6w_p6_gdn__gdn_lorma.pt', {'REP_PENALTY': 1.2}, {}, 0),
    ('lorma_reppen1.2_norep3', 'p6w_p6_gdn__gdn_lorma.pt', {'REP_PENALTY': 1.2, 'NO_REPEAT_NGRAM': 3}, {}, 0),
    ('lorma_chunks2', 'p6w_p6_gdn__gdn_lorma.pt', {}, {}, 2),
    ('lorma_norep3_chunks2', 'p6w_p6_gdn__gdn_lorma.pt', {'NO_REPEAT_NGRAM': 3}, {}, 2),
    ('lorma_len_300_450', 'p6w_p6_gdn__gdn_lorma.pt', {'MIN_NEW': 300, 'MAX_NEW': 450}, {}, 0),
    ('ft_norep3', 'p6w_p6_gdn2__gdn_full_ft.pt', {'NO_REPEAT_NGRAM': 3}, {}, 0),
    ('ft_norep3_sample', 'p6w_p6_gdn2__gdn_full_ft.pt', {'NO_REPEAT_NGRAM': 3, 'STRATEGY': 'sample', 'TEMPERATURE': 0.7, 'TOP_P': 0.9}, {}, 0),
    ('hb_norep3', 'p6w_p6_gdn2__gdn_houlsby.pt', {'NO_REPEAT_NGRAM': 3}, {}, 0),
]
DEC_BASELINES = []       # jobs that repeat the training notebook's own setting (not re-submitted)
TEST_TOP = 2                  # the best TEST_TOP other jobs (dev BERTScore) also summarise the test set -> submission zips
TEST_ALSO = []
EPOCH_CHECKPOINTS = False
MAX_SWEEP_HOURS = 10.5
QUICK_TEST = False            # True = tiny end-to-end check (few short documents, 1 epoch, short summaries, first 2 setups)
if QUICK_TEST:
    FULL_RUN = False; MAX_TRAIN_DOCS = 24; N_DEV = 8; DEV_GEN = 4
    SWEEP = [(n, dict(o, EPOCHS=1, EPOCH_DOCS=24)) for n, o in SWEEP[:2]]
    SWEEP = [(n, dict(o, KD_SUMMARIES=o["KD_SUMMARIES"].replace(".json", "_quicktest.json")) if o.get("KD_SUMMARIES") else o) for n, o in SWEEP]
    BASE.update(CONTEXT=384, SUM_MAX=64, ENC_MAX_TOK=256, ACCUM=4)
    GEN.update(MIN_NEW=8, MAX_NEW=24)
    KD_TOPK_JOBS = [(w.replace(f"p6w_{SWEEP_NAME}__", f"p6w_{SWEEP_NAME}_quicktest__"), o.replace(".pt", "_quicktest.pt"), k) for w, o, k in KD_TOPK_JOBS]
    SWEEP = [(n, dict(o, KD_TOPK=o["KD_TOPK"].replace(".pt", "_quicktest.pt")) if o.get("KD_TOPK") else o) for n, o in SWEEP]
    KD_JOBS = [(w.replace(f"p6w_{SWEEP_NAME}__", f"p6w_{SWEEP_NAME}_quicktest__"), o.replace(".json", "_quicktest.json"), 4) for w, o, n in KD_JOBS]
    DECODE = [(n, w.replace("p6w_", "p6w_").replace("__", "_quicktest__", 1), g, c, k) for n, w, g, c, k in DECODE[:3]]
    DECODE = [(n, w, dict(g, MIN_NEW=min(g.get("MIN_NEW", 8), 8), MAX_NEW=min(g.get("MAX_NEW", 24), 24)), c, k) for n, w, g, c, k in DECODE]
    SWEEP_NAME += "_quicktest"
print(f"{SWEEP_NAME}: {len(SWEEP)} setups:", [n for n, _ in SWEEP])

p6_decode_gdn: 0 setups: []


## 2. Set-up

In [2]:
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

import subprocess as _sp
_sp.run([sys.executable, "-m", "pip", "install", "-q", "bert-score", "rouge-score"], check=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 30.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.4.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.4.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.4.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.4.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 37.0 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'bert-score', 'rouge-score'], returncode=0)

## 3. Phase-1 BPE tokenizer

In [3]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)



bpe_tokenizer = BPETokenizer.load(fetch_model_file(TOKENIZER_NAME))
VOCAB = bpe_tokenizer.get_vocab_size()
print("tokenizer:", TOKENIZER_NAME, "| vocab", VOCAB)

(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json in the old repo kavincg23/cs781-checkpoints)
tokenizer: a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json | vocab 40000


## 4. Phase-1 model code (unchanged)

In [4]:
NORM_TYPE = "layernorm"; NORM_POSITION = "pre"   # as in the Phase-1 run
CONTEXT_LEN = 2048
ENC_MAX = 1024; ENC_MIN = 384; DEC_LEN = 384   # Phase-1 shapes (the encoder read up to 1024-token prompts)

import math
class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# ---- Settings specific to this notebook (encoder-decoder) ---------------------------------------------------
GQA_GROUP_SIZE = 4      # query heads that share one key/value head (12 query heads -> 3 KV heads), in every attention layer
USE_NATIVE_GQA = False  # (kept for symmetry with the decoder-only notebook: K/V are expanded with repeat_interleave)
POSITIONAL_ENCODING = "rope"
assert POSITIONAL_ENCODING == "rope", "this notebook is RoPE-only"
ROPE_BASE = 10000.0

PAD_ID = 0              # "<pad>" is the tokenizer's first special token
MASK_ID = 2             # "<s>" is never produced by the text stream, so it doubles as the [MASK] token of the encoder loss

# Shapes (ENC_MAX / ENC_MIN / DEC_LEN) are defined in the training-examples cell, which runs before this one.
ENC_LAYOUT = ["bigdn"] * 6   # "bigdn" = bidirectional Gated DeltaNet block, "attn" = bidirectional GQA + RoPE block (see EncoderBlock)
N_ENC_LAYERS = len(ENC_LAYOUT)
N_DEC_LAYERS = 4        # the decoder: causal GQA self-attention + cross-attention to the prompt + MLP
assert ENC_MAX + DEC_LEN <= CONTEXT_LEN, "RoPE tables are CONTEXT_LEN long: ENC_MAX + DEC_LEN must fit"

# Optional extra loss on the encoder (BERT-style masked tokens). Only a fraction of each batch is corrupted, so the decoder
# mostly still sees clean prompts. ENC_MLM = False gives the plain prefix-LM objective.
ENC_MLM = True
MLM_BATCH_FRACTION = 0.25   # share of the examples in a batch whose encoder input is corrupted
MLM_MASK_RATE = 0.15        # share of those examples' prompt tokens that are selected (80% -> [MASK], 10% random, 10% kept)
MLM_WEIGHT = 0.5            # weight of the masked-token loss next to the decoder's next-token loss

# --- Novel modifications (beyond vanilla GDN encoder-decoder) ---
# 3. ENCODER LAYER MIXING: instead of using only the final encoder layer output for
#    cross-attention, each decoder layer attends to a LEARNED WEIGHTED COMBINATION of
#    all encoder layer outputs. Inspired by ELMo (Peters et al., 2018): different
#    encoder depths capture different features. Shallow GDN layers may recognise local
#    legal phrases ("under section 482"); deeper layers capture discourse structure and
#    argument flow. Each decoder layer learns which encoder depths matter most for its
#    role -- early decoder layers may weight shallow encoder features for lexical
#    grounding, while later decoder layers may weight deeper encoder features for
#    coherent continuation.
ENCODER_LAYER_MIXING = True   # False = use only final encoder output (vanilla)

# 4. GATED CROSS-ATTENTION: each decoder layer scales its cross-attention output by a
#    learned scalar gate before adding to the residual stream. In a continuation task
#    the decoder must balance faithfulness to the prompt (encoder) against fluency
#    (self-attention + FFN). Not every decoder layer needs the same amount of encoder
#    context: early layers may need strong prompt grounding, while later layers may rely
#    on the autoregressive context for surface fluency.
CROSS_GATE = True             # False = vanilla (all cross-attention residuals at 1.0)
CROSS_GATE_INIT = 0.5         # initial gate (0.5 = half the encoder signal initially)


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


import math

GDN_HEADS = 4          # heads per Gated DeltaNet layer
GDN_HEAD_DIM = 128     # key dim = value dim per head (inner width = 4 x 128 = 512)
GDN_CONV = 4           # width of the short causal depthwise convolution over q, k, v
GDN_CHUNK = 64         # tokens per chunk in the chunkwise form


def _l2norm(x, eps=1e-6):
    return x / (x.norm(dim=-1, keepdim=True) + eps)


def gdn_recurrent(q, k, v, log_alpha, beta, state=None):
    """REFERENCE (slow, one token at a time). q,k: (B,H,L,dk); v: (B,H,L,dv); log_alpha, beta: (B,H,L); state: (B,H,dv,dk) or None.
    Returns outputs (B,H,L,dv) and the final state."""
    B, H, L, dk = q.shape
    dv = v.shape[-1]
    S = q.new_zeros(B, H, dv, dk) if state is None else state
    outs = []
    for t in range(L):
        kt, vt, qt = k[:, :, t], v[:, :, t], q[:, :, t]
        a = log_alpha[:, :, t].exp()[..., None, None]
        b = beta[:, :, t][..., None, None]
        Sk = torch.einsum("bhvk,bhk->bhv", S, kt)
        S = a * (S - b * Sk.unsqueeze(-1) * kt.unsqueeze(-2)) + b * vt.unsqueeze(-1) * kt.unsqueeze(-2)
        outs.append(torch.einsum("bhvk,bhk->bhv", S, qt))
    return torch.stack(outs, dim=2), S


def gdn_chunkwise(q, k, v, log_alpha, beta, state=None, chunk=64):
    """Same function as gdn_recurrent, computed `chunk` tokens at a time.
    Inside a chunk (c_i = cumulative log-decay, gamma_i = exp(c_i)) the state is  S_i = gamma_i S0 + sum_{j<=i} (gamma_i/gamma_j) d_j k_j^T,
    where the 'corrected values' d_j solve a small lower-triangular system:  (I + A) D = beta*V - (beta*gamma*K) S0^T ,
    A_ij = beta_i (gamma_i/gamma_j) k_i.k_j for j < i.   With T = (I+A)^-1:  U = T (beta V),  W = T (beta gamma K),  D = U - W S0^T.
    Then   O = diag(gamma) Q S0^T + ((Q K^T) * Lambda) D ,   S_end = gamma_C S0 + D^T diag(gamma_C/gamma) K.
    Decay ratios are built as exp(c_i - c_j) AFTER masking j > i (never a division of cumulative products), so nothing over/underflows."""
    B, H, L, dk = q.shape
    dv = v.shape[-1]
    pad = (-L) % chunk
    if pad:   # right-pad with 'do nothing' tokens (k = v = q = 0, beta = 0, alpha = 1): the state is untouched by them
        q, k, v = (F.pad(x, (0, 0, 0, pad)) for x in (q, k, v))
        log_alpha, beta = (F.pad(x, (0, pad)) for x in (log_alpha, beta))
    N = (L + pad) // chunk
    q, k, v = (x.reshape(B, H, N, chunk, x.shape[-1]) for x in (q, k, v))
    la, be = (x.reshape(B, H, N, chunk) for x in (log_alpha, beta))
    c = la.cumsum(dim=-1)                                                         # (B,H,N,C) inclusive cumulative log-decay
    on_or_below = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device))
    strictly_below = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device), diagonal=-1)
    lam = (c.unsqueeze(-1) - c.unsqueeze(-2)).masked_fill(~on_or_below, float("-inf")).exp()   # gamma_i/gamma_j for j<=i, else 0
    A = (be.unsqueeze(-1) * (k @ k.transpose(-1, -2)) * lam).masked_fill(~strictly_below, 0.0)
    eye = torch.eye(chunk, dtype=q.dtype, device=q.device)
    T = torch.linalg.solve_triangular(eye + A, eye.expand_as(A), upper=False)      # (I + A)^-1
    u = T @ (be.unsqueeze(-1) * v)                                                 # (B,H,N,C,dv)
    w = T @ ((be * c.exp()).unsqueeze(-1) * k)                                     # (B,H,N,C,dk)
    qk = (q @ k.transpose(-1, -2)) * lam                                           # (B,H,N,C,C)
    q_g = c.exp().unsqueeze(-1) * q
    c_last = c[..., -1]                                                            # (B,H,N)
    k_dec = (c_last.unsqueeze(-1) - c).exp().unsqueeze(-1) * k
    S = q.new_zeros(B, H, dv, dk) if state is None else state
    outs = []
    for n in range(N):
        St = S.transpose(-1, -2)                                                   # (B,H,dk,dv)
        d = u[:, :, n] - w[:, :, n] @ St                                           # corrected values (B,H,C,dv)
        outs.append(q_g[:, :, n] @ St + qk[:, :, n] @ d)
        S = c_last[:, :, n].exp()[..., None, None] * S + d.transpose(-1, -2) @ k_dec[:, :, n]
    o = torch.stack(outs, dim=2).reshape(B, H, N * chunk, dv)[:, :, :L]
    return o, S


class GatedDeltaNetMixer(nn.Module):
    """The sequence-mixing part of a Gated DeltaNet layer (the block wraps it with the norm, residual and MLP).
    x -> q,k,v projections -> short causal depthwise conv + SiLU -> L2-normalised q,k -> delta-rule memory (gate alpha, step beta)
      -> per-head RMSNorm -> times SiLU(output gate) -> output projection.   Always computed in fp32 (the recurrence is the numerically
    sensitive part; it is a small share of the compute)."""

    def __init__(self, d_model, n_heads=GDN_HEADS, head_dim=GDN_HEAD_DIM, conv_width=GDN_CONV, chunk=GDN_CHUNK):
        super().__init__()
        self.n_heads, self.head_dim, self.conv_width, self.chunk = n_heads, head_dim, conv_width, chunk
        inner = n_heads * head_dim
        self.inner = inner
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.g_proj = nn.Linear(d_model, inner, bias=False)          # output gate
        self.o_proj = nn.Linear(inner, d_model, bias=False)
        self.a_proj = nn.Linear(d_model, n_heads, bias=False)        # -> decay alpha_t (per head)
        self.b_proj = nn.Linear(d_model, n_heads, bias=False)        # -> step size beta_t (per head)
        self.conv = nn.Conv1d(3 * inner, 3 * inner, conv_width, groups=3 * inner, bias=False)   # depthwise, applied to [q;k;v]
        self.out_norm = RMSNorm(head_dim)
        # decay parameterisation (Mamba-2 style): alpha = exp(-exp(A_log) * softplus(a_proj(x) + dt_bias)), A in [1,16], dt in [1e-3, 1e-1]
        self.A_log = nn.Parameter(torch.log(torch.empty(n_heads).uniform_(1.0, 16.0)))
        dt = torch.exp(torch.rand(n_heads) * (math.log(0.1) - math.log(0.001)) + math.log(0.001))
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))        # inverse softplus of dt

    def _project(self, x):
        return torch.cat([self.q_proj(x), self.k_proj(x), self.v_proj(x)], dim=-1)

    def _split_heads(self, qkv_act):
        B, L, _ = qkv_act.shape
        q, k, v = qkv_act.split(self.inner, dim=-1)
        sh = lambda t: t.reshape(B, L, self.n_heads, self.head_dim).transpose(1, 2)    # (B,H,L,hd)
        q, k, v = sh(q), sh(k), sh(v)
        return _l2norm(q) * (self.head_dim ** -0.5), _l2norm(k), v

    def _decay_and_step(self, x):
        beta = torch.sigmoid(self.b_proj(x))                                            # (..., H)
        log_alpha = -self.A_log.exp() * F.softplus(self.a_proj(x) + self.dt_bias)       # (..., H), <= 0
        return log_alpha, beta

    def _finish(self, o, x):
        """o: (B,H,L,hd) memory read-out -> per-head norm -> output gate -> output projection."""
        B, H, L, hd = o.shape
        o = self.out_norm(o.transpose(1, 2))                                            # (B,L,H,hd)
        gate = F.silu(self.g_proj(x)).reshape(B, L, H, hd)
        return self.o_proj((o * gate).reshape(B, L, H * hd))

    def forward(self, x, key_padding_mask=None, return_state=False, use_recurrent=False):
        """x: (B,L,D). key_padding_mask (optional, (B,L) bool, True = real token): LEFT-padding positions are made 'do nothing' tokens
        (zero input, beta = 0, alpha = 1) so the state stays exactly zero until the first real token, whatever the padding."""
        dtype_in = x.dtype
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            B, L, _ = x.shape
            if key_padding_mask is not None:
                x = x * key_padding_mask.unsqueeze(-1).to(x.dtype)
            qkv = self._project(x)                                                      # (B,L,3*inner) pre-conv
            conv_out = self.conv(F.pad(qkv.transpose(1, 2), (self.conv_width - 1, 0)))  # causal: left zero-pad, output length L
            q, k, v = self._split_heads(F.silu(conv_out.transpose(1, 2)))
            log_alpha, beta = self._decay_and_step(x)                                   # (B,L,H)
            if key_padding_mask is not None:
                m = key_padding_mask.unsqueeze(-1).to(x.dtype)
                log_alpha, beta = log_alpha * m, beta * m
            log_alpha, beta = log_alpha.transpose(1, 2), beta.transpose(1, 2)           # (B,H,L)
            if use_recurrent:
                o, S = gdn_recurrent(q, k, v, log_alpha, beta)
            else:
                o, S = gdn_chunkwise(q, k, v, log_alpha, beta, chunk=self.chunk)
            out = self._finish(o, x)
            if return_state:
                keep = self.conv_width - 1                                              # last pre-conv rows = the decode conv buffer
                buf = qkv[:, -keep:]
                if buf.shape[1] < keep:
                    buf = F.pad(buf, (0, 0, keep - buf.shape[1], 0))
                return out.to(dtype_in), S, buf
        return out.to(dtype_in)

    def step(self, x_t, S, conv_buf):
        """ONE new token per row. x_t: (B,D); S: (B,H,hd,hd) memory; conv_buf: (B,conv_width-1,3*inner) last pre-conv rows.
        Returns (out (B,D), new S, new conv_buf). O(1) work and memory per token."""
        dtype_in = x_t.dtype
        with torch.autocast(device_type=x_t.device.type, enabled=False):
            x = x_t.float()
            qkv_t = self._project(x)                                                    # (B,3*inner)
            window = torch.cat([conv_buf, qkv_t.unsqueeze(1)], dim=1)                   # (B,conv_width,3*inner), oldest first
            w = self.conv.weight.squeeze(1)                                             # (3*inner, conv_width)
            conv_out = (window * w.t().unsqueeze(0)).sum(dim=1)                         # same as the causal conv at this position
            q, k, v = self._split_heads(F.silu(conv_out).unsqueeze(1))                  # (B,H,1,hd)
            q, k, v = q[:, :, 0], k[:, :, 0], v[:, :, 0]                                # (B,H,hd)
            log_alpha, beta = self._decay_and_step(x)                                   # (B,H)
            a, b = log_alpha.exp()[..., None, None], beta[..., None, None]
            Sk = torch.einsum("bhvk,bhk->bhv", S, k)
            S = a * (S - b * Sk.unsqueeze(-1) * k.unsqueeze(-2)) + b * v.unsqueeze(-1) * k.unsqueeze(-2)
            o = torch.einsum("bhvk,bhk->bhv", S, q).unsqueeze(2)                        # (B,H,1,hd)
            out = self._finish(o, x.unsqueeze(1))[:, 0]
        return out.to(dtype_in), S, window[:, 1:]


class BiGDNEncoderBlock(nn.Module):
    """ENCODER block built on Gated DeltaNet. The memory update is causal (left to right), so to let every prompt token see the whole
    prompt the block runs TWO mixers: one reads the prompt left-to-right, the other right-to-left (the sequence is flipped, mixed
    causally, and flipped back); their outputs are added. Pre-norm residual block with the same GELU MLP as every other block.
    Cost is linear in the prompt length. Left padding is harmless: pad positions are 'do nothing' tokens in both directions."""

    def __init__(self, d_model, d_ff):
        super().__init__()
        assert NORM_POSITION == "pre" and NORM_TYPE != "deepnorm", "written for pre-norm blocks"
        self.fwd = GatedDeltaNetMixer(d_model)
        self.bwd = GatedDeltaNetMixer(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1, self.norm2 = make_norm(d_model, NORM_TYPE), make_norm(d_model, NORM_TYPE)

    def forward(self, x, rope, key_padding_mask):          # rope is unused: the recurrence carries position implicitly
        h = self.norm1(x)
        y = self.fwd(h, key_padding_mask)
        flipped_mask = None if key_padding_mask is None else key_padding_mask.flip(1)
        y = y + self.bwd(h.flip(1), flipped_mask).flip(1)
        x = x + y
        return x + self.ff(self.norm2(x))


class SelfAttention(nn.Module):
    """Grouped-query self-attention with RoPE. causal=False: the ENCODER (every prompt token sees the whole prompt).
    causal=True: the DECODER (parallel form for training; decode_step() for one token with a KV cache)."""

    def __init__(self, d_model, n_heads, n_kv_heads, causal):
        super().__init__()
        self.n_heads, self.n_kv_heads, self.causal = n_heads, n_kv_heads, causal
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, rope, key_padding_mask=None, k_cache=None, v_cache=None):
        """x: (B, S, d). rope: (cos, sin), each (B, 1, S, head_dim). key_padding_mask: (B, S) bool, True = real token (encoder only).
        If k_cache/v_cache are given the (rotated) keys/values are also written to positions [0:S]."""
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, *rope), apply_rope(k, *rope)
        if k_cache is not None:
            k_cache[:, :, :S] = k.to(k_cache.dtype)
            v_cache[:, :, :S] = v.to(v_cache.dtype)
        if self.group_size > 1:
            k = k.repeat_interleave(self.group_size, dim=1)
            v = v.repeat_interleave(self.group_size, dim=1)
        if self.causal:
            assert key_padding_mask is None, "decoder sequences are never padded"
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            mask = None if key_padding_mask is None else key_padding_mask.view(B, 1, 1, S)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, S, self.n_heads * self.head_dim))

    def decode_step(self, x_t, rope_t, k_cache, v_cache, cache_len):
        """ONE new decoder position per row. x_t: (B, d); rope_t: (cos, sin) each (B, head_dim). Writes K/V at index cache_len."""
        B = x_t.shape[0]
        q = self.q_proj(x_t).view(B, self.n_kv_heads, self.group_size, self.head_dim)     # query heads folded into the query axis
        k_new = self.k_proj(x_t).view(B, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(B, self.n_kv_heads, self.head_dim)
        cos, sin = rope_t
        q = apply_rope(q, cos.view(B, 1, 1, -1), sin.view(B, 1, 1, -1))
        k_new = apply_rope(k_new, cos.view(B, 1, -1), sin.view(B, 1, -1))
        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k, v = k_cache[:, :, :cache_len + 1], v_cache[:, :, :cache_len + 1]
        out = F.scaled_dot_product_attention(q.to(k.dtype), k, v)                           # (B, n_kv, group, head_dim)
        return self.out_proj(out.reshape(B, self.n_heads * self.head_dim).to(x_t.dtype))


class CrossAttention(nn.Module):
    """Decoder -> encoder attention (GQA). Queries come from the decoder, keys/values from the encoder output, which are computed
    ONCE per prompt (kv()) and reused at every decoding step.

    Positions: the prompt tokens sit at positions 0..L-1 and the decoder token number i sits at position L-1+i (the decoder's first
    input is the prompt's last token). RoPE is applied to the queries (decoder positions) and to the keys (prompt positions), so
    the attention score depends on how far back in the PROMPT a token is -- 'the sentence that just ended' is distinguishable from
    'a clause 600 tokens ago', which a plain position-free cross-attention cannot tell."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def kv(self, enc_out, enc_rope):
        B, S, _ = enc_out.shape
        k = self.k_proj(enc_out).view(B, S, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.n_kv_heads, self.head_dim).transpose(1, 2)
        return apply_rope(k, *enc_rope), v

    def forward(self, x, enc_k, enc_v, q_rope, enc_mask):
        """x: (B, T, d) decoder states. enc_k/enc_v: (B, n_kv, S, head_dim). q_rope: (cos, sin) each (B, 1, T, head_dim).
        enc_mask: (B, S) bool, True = real prompt token (None if the batch has no padding)."""
        B, T, _ = x.shape
        q = apply_rope(self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2), *q_rope)
        k, v = enc_k, enc_v
        if self.group_size > 1:
            k = k.repeat_interleave(self.group_size, dim=1)
            v = v.repeat_interleave(self.group_size, dim=1)
        mask = None if enc_mask is None else enc_mask.view(B, 1, 1, -1)
        out = F.scaled_dot_product_attention(q.to(k.dtype), k, v, attn_mask=mask)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim).to(x.dtype))

    def decode_step(self, x_t, enc_k, enc_v, rope_t, enc_mask):
        B = x_t.shape[0]
        q = self.q_proj(x_t).view(B, self.n_kv_heads, self.group_size, self.head_dim)
        cos, sin = rope_t
        q = apply_rope(q, cos.view(B, 1, 1, -1), sin.view(B, 1, 1, -1))
        mask = None if enc_mask is None else enc_mask.view(B, 1, 1, -1)
        out = F.scaled_dot_product_attention(q.to(enc_k.dtype), enc_k, enc_v, attn_mask=mask)
        return self.out_proj(out.reshape(B, self.n_heads * self.head_dim).to(x_t.dtype))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = SelfAttention(d_model, n_heads, max(1, n_heads // GQA_GROUP_SIZE), causal=False)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1, self.norm2 = make_norm(d_model, NORM_TYPE), make_norm(d_model, NORM_TYPE)

    def forward(self, x, rope, key_padding_mask):
        x = x + self.attn(self.norm1(x), rope, key_padding_mask=key_padding_mask)
        return x + self.ff(self.norm2(x))


class DecoderBlock(nn.Module):
    """causal self-attention -> cross-attention over the prompt -> feed-forward (pre-norm, residual around each)."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        n_kv = max(1, n_heads // GQA_GROUP_SIZE)
        self.self_attn = SelfAttention(d_model, n_heads, n_kv, causal=True)
        self.cross_attn = CrossAttention(d_model, n_heads, n_kv)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1, self.norm2, self.norm3 = (make_norm(d_model, NORM_TYPE) for _ in range(3))
        # Novel: gated cross-attention residual
        if CROSS_GATE:
            self.cross_gate = nn.Parameter(torch.tensor(CROSS_GATE_INIT))
        else:
            self.register_buffer('cross_gate', torch.tensor(1.0))

    def forward(self, x, rope, enc_k, enc_v, enc_mask, k_cache=None, v_cache=None):
        x = x + self.self_attn(self.norm1(x), rope, k_cache=k_cache, v_cache=v_cache)
        x = x + self.cross_gate * self.cross_attn(self.norm2(x), enc_k, enc_v, rope, enc_mask)
        return x + self.ff(self.norm3(x))

    def decode_step(self, x_t, rope_t, enc_k, enc_v, enc_mask, k_cache, v_cache, cache_len):
        x_t = x_t + self.self_attn.decode_step(self.norm1(x_t), rope_t, k_cache, v_cache, cache_len)
        x_t = x_t + self.cross_gate * self.cross_attn.decode_step(self.norm2(x_t), enc_k, enc_v, rope_t, enc_mask)
        return x_t + self.ff(self.norm3(x_t))


class EncDecCache:
    """Everything generation needs per batch: the cross-attention keys/values of every decoder layer (computed once from the
    prompt), the decoder's own self-attention KV cache, and the prompt padding mask."""

    def __init__(self, model, batch_size, device, dtype):
        n_kv, hd = model.n_kv_heads, model.head_dim
        self.self_k = [torch.zeros(batch_size, n_kv, DEC_LEN, hd, device=device, dtype=dtype) for _ in model.dec_blocks]
        self.self_v = [torch.zeros_like(t) for t in self.self_k]
        self.cross_k, self.cross_v = [], []
        self.enc_mask = None
        self.start_pos = None     # (B,) long: position of the decoder's first input token (= prompt length - 1)
        self.length = 0           # decoder positions filled so far
        self.max_len = DEC_LEN

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.self_k + self.self_v + self.cross_k + self.cross_v)


class EncoderDecoderLM(nn.Module):
    """Prefix-LM encoder-decoder. The ENCODER (bidirectional, RoPE) reads the prompt; the DECODER (causal, RoPE) writes the continuation,
    attending to its own past and, through cross-attention, to the encoder output. The token embedding is shared by encoder and
    decoder; the output projection is separate (as in the decoder-only notebook)."""

    def __init__(self, vocab_size, d_model, n_heads, enc_layout, n_dec_layers, d_ff, context_len):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.token_emb = nn.Embedding(vocab_size, d_model)
        cos, sin = build_rope_cache(context_len, self.head_dim, ROPE_BASE)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.enc_blocks = nn.ModuleList([BiGDNEncoderBlock(d_model, d_ff) if kind == "bigdn" else EncoderBlock(d_model, n_heads, d_ff) for kind in enc_layout])
        self.enc_norm = make_norm(d_model, NORM_TYPE)
        self.dec_blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_dec_layers)])
        self.dec_norm = make_norm(d_model, NORM_TYPE)
        self.fc_out = nn.Linear(d_model, vocab_size)
        # Novel: encoder layer mixing (each decoder layer learns its own weighted view of encoder layers)
        n_enc = len(enc_layout)
        if ENCODER_LAYER_MIXING:
            init_w = torch.full((n_dec_layers, n_enc), -10.0)
            init_w[:, -1] = 0.0   # init: only the final encoder layer contributes (= vanilla)
            self.enc_mix_logits = nn.Parameter(init_w)
        else:
            self.enc_mix_logits = None

    def _rope(self, positions):
        """positions (B, S) long -> (cos, sin), each (B, 1, S, head_dim)."""
        return self.rope_cos[positions].unsqueeze(1), self.rope_sin[positions].unsqueeze(1)

    def encode(self, enc_ids, enc_mask, enc_pos):
        """enc_ids (B, S) left-padded prompt ids; enc_mask (B, S) bool or None; enc_pos (B, S) positions (first real token = 0).
        Returns the final encoder states (B, S, d) and the encoder RoPE (needed for the cross-attention keys)."""
        rope = self._rope(enc_pos)
        h = self.token_emb(enc_ids)
        if self.enc_mix_logits is not None:
            self._enc_intermediates = []
        for block in self.enc_blocks:
            h = block(h, rope, enc_mask)
            if self.enc_mix_logits is not None:
                self._enc_intermediates.append(h)
        return self.enc_norm(h), rope

    def forward(self, enc_ids, dec_ids, enc_mask, enc_pos, dec_pos, last_position_only=False, return_enc=False):
        """Training / reference forward. dec_ids (B, T): the decoder INPUT tokens (the prompt's last token, then the continuation
        shifted right); dec_pos (B, T): their positions (prompt_length - 1 + i). Returns logits (B, T, V) over the next tokens."""
        enc_h, enc_rope = self.encode(enc_ids, enc_mask, enc_pos)
        rope = self._rope(dec_pos)
        h = self.token_emb(dec_ids)
        if self.enc_mix_logits is not None:
            stacked_enc = torch.stack(self._enc_intermediates, dim=0)  # (n_enc, B, S, d)
        for i, block in enumerate(self.dec_blocks):
            if self.enc_mix_logits is not None:
                w = F.softmax(self.enc_mix_logits[i], dim=0)          # (n_enc,)
                mixed = (stacked_enc * w.view(-1, 1, 1, 1)).sum(0)    # (B, S, d)
                k, v = block.cross_attn.kv(self.enc_norm(mixed), enc_rope)
            else:
                k, v = block.cross_attn.kv(enc_h, enc_rope)
            h = block(h, rope, k, v, enc_mask)
        h = self.dec_norm(h)
        if last_position_only:
            h = h[:, -1, :]
        logits = self.fc_out(h)
        return (logits, enc_h) if return_enc else logits

    def mlm_logits(self, enc_states):
        """Masked-token prediction from encoder states, through the (tied) token embedding matrix: no extra parameters."""
        return F.linear(enc_states, self.token_emb.weight)

    # ---------------- cached generation ----------------
    def start_generation(self, enc_ids, pad_lens, cache):
        """Run the encoder ONCE, store every decoder layer's cross-attention K/V, then feed the decoder's first input token (the
        prompt's last token) and return the logits for the first generated token (B, V)."""
        B, S = enc_ids.shape
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=enc_ids.device)
        arange = torch.arange(S, device=enc_ids.device).unsqueeze(0)
        enc_pos = (arange - pad_t.unsqueeze(1)).clamp(min=0)
        cache.enc_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None
        cache.start_pos = (S - pad_t - 1)
        enc_h, enc_rope = self.encode(enc_ids, cache.enc_mask, enc_pos)
        cache.cross_k, cache.cross_v = [], []
        if self.enc_mix_logits is not None:
            stacked_enc = torch.stack(self._enc_intermediates, dim=0)
        for i, block in enumerate(self.dec_blocks):
            if self.enc_mix_logits is not None:
                w = F.softmax(self.enc_mix_logits[i], dim=0)
                mixed = (stacked_enc * w.view(-1, 1, 1, 1)).sum(0)
                k, v = block.cross_attn.kv(self.enc_norm(mixed), enc_rope)
            else:
                k, v = block.cross_attn.kv(enc_h, enc_rope)
            cache.cross_k.append(k.to(cache.self_k[0].dtype))
            cache.cross_v.append(v.to(cache.self_k[0].dtype))
        cache.length = 0
        return self.decode_step(enc_ids[:, -1], cache)

    def decode_step(self, token_ids, cache):
        """ONE decoder token per row (token_ids (B,)). Appends to the self-attention cache; returns logits (B, V) for the NEXT token."""
        assert cache.length < cache.max_len, "decoder cache is full"
        pos = cache.start_pos + cache.length                                   # (B,)
        rope_t = (self.rope_cos[pos], self.rope_sin[pos])
        h = self.token_emb(token_ids)
        for i, block in enumerate(self.dec_blocks):
            h = block.decode_step(h, rope_t, cache.cross_k[i], cache.cross_v[i], cache.enc_mask,
                                  cache.self_k[i], cache.self_v[i], cache.length)
        cache.length += 1
        return self.fc_out(self.dec_norm(h))

## 5. Backbone wrappers (unchanged from Phase 2)

In [5]:
# ===================== NER backbone wrappers for the other Phase-1 architectures =====================
# The Phase-1 model code above is UNCHANGED. Each wrapper only exposes the hidden states that feed the NER head.

def _unique(mods):
    seen, out = set(), []
    for m in mods:
        if id(m) not in seen:
            seen.add(id(m)); out.append(m)
    return out


class ForwardBackbone(nn.Module):
    """For decoder models with their own forward logic (MoR recursion + routing, Boundary Canon): the model's own forward()
    is called with the vocabulary projection replaced by the identity, so it returns its last hidden states."""

    def __init__(self, lm):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.lm = lm
        self.causal = True
        if isinstance(lm.fc_out, nn.Sequential):
            self.final_norm = lm.fc_out[0]
        else:
            self.final_norm = nn.Identity()
        lm.fc_out = nn.Identity()
        self.d_model = lm.token_emb.embedding_dim

    def layer_list(self):
        return _unique(list(self.lm.blocks))

    def final_modules_params(self):
        return list(self.final_norm.parameters())

    def embedding_weight(self):
        return self.lm.token_emb.weight

    def adapter_sites(self, kind):
        blocks = _unique(list(self.lm.blocks))
        if kind == "houlsby":
            return [(m, self.d_model) for m in _unique([x for b in blocks for x in (b.attn, b.ff)])]
        if kind == "pfeiffer":
            return [(m, self.d_model) for m in _unique([b.ff for b in blocks])]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        raise NotImplementedError("prefix tuning is only wired for the plain GQA/MoE decoders")

    def features(self, ids, mask, soft_prompt=None):
        assert soft_prompt is None, "prompt tuning is only wired for the plain GQA/MoE decoders"
        # right padding + causal model: real tokens never see the pads, so no mask is passed (as in Phase-1 training)
        return self.final_norm(self.lm(ids))


class LSTMBackbone(nn.Module):
    """The Phase-1 3-layer LSTM, split into one nn.LSTM per layer (weights copied exactly) so that adapters can sit between
    layers and partial fine-tuning can pick the top layers. Hidden states of the last layer (LSTM_HIDDEN wide) feed the head."""

    def __init__(self, lm):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.emb = lm.emb
        L, H = lm.lstm.num_layers, lm.lstm.hidden_size
        self.layers = nn.ModuleList()
        for k in range(L):
            layer = nn.LSTM(lm.lstm.input_size if k == 0 else H, H, num_layers=1, batch_first=True)
            with torch.no_grad():
                for n in ("weight_ih", "weight_hh", "bias_ih", "bias_hh"):
                    getattr(layer, f"{n}_l0").copy_(getattr(lm.lstm, f"{n}_l{k}"))
            self.layers.append(layer)
        self.taps = nn.ModuleList([nn.Identity() for _ in range(L)])    # adapter attachment points (after each layer)
        self.d_model = H
        self.causal = True

    def layer_list(self):
        return list(self.layers)

    def final_modules_params(self):
        return []

    def embedding_weight(self):
        return self.emb.weight

    def adapter_sites(self, kind):
        if kind == "houlsby":
            return [(t, self.d_model) for t in self.taps]
        if kind == "pfeiffer":
            return [(self.taps[-1], self.d_model)]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        raise NotImplementedError("prefix tuning does not apply to an LSTM")

    def features(self, ids, mask, soft_prompt=None):
        assert soft_prompt is None, "prompt tuning is not wired for the LSTM"
        h = self.emb(ids)
        for layer, tap in zip(self.layers, self.taps):
            layer.flatten_parameters()
            h, _ = layer(h)
            h = tap(h)
        return h


class GDNEncoderBackbone(nn.Module):
    """The ENCODER of the Phase-1 Gated-DeltaNet encoder-decoder (6 bidirectional GDN blocks + final norm). Its decoder and
    output head are dropped. Bidirectional by construction (a forward and a backward GDN mixer in every block)."""

    def __init__(self, lm):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.lm = lm
        lm.dec_blocks = nn.ModuleList()
        lm.fc_out = nn.Identity()
        lm.enc_mix_logits = None            # only used by the decoder; also stops it storing every layer's states
        self.causal = False
        self.d_model = lm.token_emb.embedding_dim

    def layer_list(self):
        return list(self.lm.enc_blocks)

    def final_modules_params(self):
        return list(self.lm.enc_norm.parameters())

    def embedding_weight(self):
        return self.lm.token_emb.weight

    def adapter_sites(self, kind):
        blocks = list(self.lm.enc_blocks)
        if kind == "houlsby":
            return [(m, self.d_model) for b in blocks for m in (b.fwd, b.bwd, b.ff)]
        if kind == "pfeiffer":
            return [(b.ff, self.d_model) for b in blocks]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        raise NotImplementedError("prefix tuning does not apply to the GDN encoder")

    def features(self, ids, mask, soft_prompt=None):
        assert soft_prompt is None, "prompt tuning is not wired for the GDN encoder"
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(B, T)
        enc_mask = None if bool(mask.all()) else mask            # the backward mixer must not read right-padding
        h, _ = self.lm.encode(ids, enc_mask, pos)
        return h


@torch.no_grad()
def gdn_sanity_loss(lm, docs, device, n_windows=6, enc_len=768, dec_len=256):
    """Phase-1 objective of the encoder-decoder on NER text: encoder reads enc_len tokens, decoder predicts the next dec_len."""
    lm.eval().to(device)
    tot, cnt = 0.0, 0
    for d in list(docs.values())[:n_windows]:
        ids = torch.tensor(d["ids"][:enc_len + dec_len], dtype=torch.long, device=device).unsqueeze(0)
        if ids.shape[1] < enc_len + 8:
            continue
        enc, rest = ids[:, :enc_len], ids[:, enc_len - 1:]
        dec_in, tgt = rest[:, :-1], rest[:, 1:]
        enc_pos = torch.arange(enc.shape[1], device=device).unsqueeze(0)
        dec_pos = (enc_len - 1 + torch.arange(dec_in.shape[1], device=device)).unsqueeze(0)
        logits = lm(enc, dec_in, None, enc_pos, dec_pos).float()
        tot += F.cross_entropy(logits.view(-1, logits.shape[-1]), tgt.reshape(-1), reduction="sum").item()
        cnt += tgt.numel()
    return tot / max(cnt, 1)


@torch.no_grad()
def lm_sanity_loss(lm, docs, device, n_windows=6, length=1024):
    """Next-token loss of the loaded Phase-1 model on a few NER-document windows (BEFORE the LM head is dropped).
    A correctly loaded model gives roughly its Phase-1 held-out loss; random weights give ~ln(vocab) = 10.6."""
    lm.eval().to(device)
    tot, cnt = 0.0, 0
    for d in list(docs.values())[:n_windows]:
        ids = torch.tensor(d["ids"][:length + 1], dtype=torch.long, device=device).unsqueeze(0)
        if ids.shape[1] < 2:
            continue
        logits = lm(ids[:, :-1]).float()
        tot += F.cross_entropy(logits.view(-1, logits.shape[-1]), ids[:, 1:].reshape(-1), reduction="sum").item()
        cnt += ids.shape[1] - 1
    return tot / max(cnt, 1)


def build_phase1_lm():
    """The Phase-1 GDN encoder-decoder exactly as trained, with its trained weights (the decoder is dropped later)."""
    torch.manual_seed(0)
    lm = EncoderDecoderLM(VOCAB, d_model=768, n_heads=12, enc_layout=ENC_LAYOUT, n_dec_layers=N_DEC_LAYERS,
                          d_ff=3072, context_len=CONTEXT_LEN)
    lm.load_state_dict(PHASE1_STATE, strict=True)
    return lm


def make_backbone(lm, C=None):
    return GDNEncoderBackbone(lm)


def phase1_sanity(lm, docs):
    return gdn_sanity_loss(lm, docs, device)        # decoder next-token loss with the encoder reading the preceding 768 tokens

## 6. Phase-6 engine: selection, examples, model wrapper, generation, scores, training

In [6]:
# ===================== Phase 6 (Abstractive Summarization) engine =====================
# Works with every Phase-1 model: decoder-only (GQA / MoE / MoR), the LSTM language model and the GDN encoder-decoder.
# A training example is  [selected document tokens] + [separator] + [summary tokens] + [</s>]  (decoder-only / LSTM: one
# sequence, loss only on the summary part) or encoder = document + separator, decoder = summary (GDN). Long documents are
# reduced to a token budget by a SELECTION strategy (head / tail / head+tail / importance = TF-IDF centrality).
import math, time, random, json, os, re, zipfile, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

PAD_ID, EOS_ID = 0, 3                    # "<pad>", "</s>" of the Phase-1 tokenizer
SENT_SPLIT = re.compile(r"(?<=[.?!;])\s+")


# ----------------------------------------------------------------------------------------------------------------------
# 1. Documents: token ids + sentence offsets; long-document selection
# ----------------------------------------------------------------------------------------------------------------------
def doc_text(x):
    return " ".join(x) if isinstance(x, list) else str(x)


def encode_with_sentences(tok, text, max_tokens):
    """Token ids of the whole text (identical to tok.encode(text): the tokenizer works word by word) + the token offset where
    every sentence starts (soff[-1] = number of tokens). Texts longer than max_tokens keep their first and last halves."""
    ids, soff = [], [0]
    for s in SENT_SPLIT.split(text):
        if s.strip():
            ids.extend(tok.encode(s)); soff.append(len(ids))
    if not ids:
        ids, soff = [1], [0, 1]
    if len(ids) > max_tokens:                     # keep head and tail, sentence offsets re-mapped
        h = max_tokens // 2; t = max_tokens - h; n = len(ids)
        new_ids = ids[:h] + ids[n - t:]
        so = [o for o in soff if o <= h] + [h + (o - (n - t)) for o in soff if o > n - t]
        ids, soff = new_ids, sorted(set(so + [len(new_ids)]))
    return np.asarray(ids, dtype=np.uint16), np.asarray(soff, dtype=np.int32)


def select_doc(d, budget, C, idf=None):
    """Up to `budget` tokens of document d, in document order. SELECT: head | tail | head_tail (HEAD_FRAC of the budget from the
    start, the rest from the end) | importance (keep the first IMP_HEAD_KEEP tokens = cause title / parties, then add whole
    sentences by TF-IDF centrality score until the budget is full)."""
    ids = d["ids"]; n = len(ids)
    if n <= budget:
        return ids
    how = C["SELECT"]
    if how == "head":
        return ids[:budget]
    if how == "tail":
        return ids[n - budget:]
    if how == "head_tail":
        h = int(budget * C.get("HEAD_FRAC", 0.5))
        return np.concatenate([ids[:h], ids[n - (budget - h):]])
    if how == "importance":
        assert idf is not None, "importance selection needs the IDF table"
        soff = d["soff"]; keep = C.get("IMP_HEAD_KEEP", 128)
        tf = np.bincount(ids.astype(np.int64), minlength=len(idf)).astype(np.float32)
        w = np.log1p(tf) * idf
        spans = [(int(soff[i]), int(soff[i + 1])) for i in range(len(soff) - 1) if soff[i + 1] > soff[i]]
        chosen, used = set(), 0
        for k, (s, e) in enumerate(spans):            # always the start of the document
            if s < keep and used + (e - s) <= budget:
                chosen.add(k); used += e - s
        score = [(w[ids[s:e]].sum() / math.sqrt(e - s), k) for k, (s, e) in enumerate(spans) if k not in chosen]
        for sc, k in sorted(score, reverse=True):
            s, e = spans[k]
            if used + (e - s) <= budget:
                chosen.add(k); used += e - s
            if used >= budget - 8:
                break
        out = np.concatenate([ids[spans[k][0]:spans[k][1]] for k in sorted(chosen)]) if chosen else ids[:budget]
        return out[:budget]
    if how == "sections":
        # section-aware (cue-based): keep the cause title (first IMP_HEAD_KEEP tokens) and the closing part (SECTION_TAIL_FRAC of the
        # budget), and fill the rest with the sentences that contain the most legal-structure cue words (issue, held, contended,
        # section, article, appeal, dismissed, ...), in document order. Cue token ids: SECTION_CUE_IDS (set from the tokenizer).
        soff = d["soff"]; keep = C.get("IMP_HEAD_KEEP", 128); tail = int(budget * C.get("SECTION_TAIL_FRAC", 0.2))
        spans = [(int(soff[i]), int(soff[i + 1])) for i in range(len(soff) - 1) if soff[i + 1] > soff[i]]
        cue = np.zeros(max(int(ids.max()) + 1, 1), dtype=bool); cue[[c for c in SECTION_CUE_IDS if c < len(cue)]] = True
        t0 = n - tail; chosen, used = set(), 0
        for k, (s_, e_) in enumerate(spans):
            if s_ < keep and used + (e_ - s_) <= budget - tail: chosen.add(k); used += e_ - s_
        score = [(cue[ids[s_:e_]].sum() / math.sqrt(e_ - s_), -k, k) for k, (s_, e_) in enumerate(spans) if k not in chosen and e_ <= t0]
        for sc, _, k in sorted(score, reverse=True):
            s_, e_ = spans[k]
            if sc <= 0: break
            if used + (e_ - s_) <= budget - tail: chosen.add(k); used += e_ - s_
        body = np.concatenate([ids[spans[k][0]:spans[k][1]] for k in sorted(chosen)]) if chosen else ids[:0]
        return np.concatenate([body[:budget - tail], ids[t0:]])[:budget]
    raise ValueError(how)


SECTION_CUES = ("held", "hold", "holding", "issue", "issues", "question", "questions", "contended", "contention", "submitted", "argued", "facts", "fact",
                "section", "sections", "article", "articles", "act", "rule", "rules", "appeal", "appellant", "respondent", "petition", "petitioner",
                "dismissed", "allowed", "quashed", "set", "aside", "convicted", "acquitted", "sentence", "order", "decree", "therefore", "accordingly",
                "conclusion", "conclude", "opinion", "reasons", "court", "high", "tribunal", "challenged", "validity", "constitution")
SECTION_CUE_IDS = []
def set_section_cues(tok, vocab):
    """Token ids whose text (lower-cased, punctuation stripped) is a cue word."""
    global SECTION_CUE_IDS
    ids = []
    for i in range(4, vocab):
        t = re.sub(r"[^a-z]", "", tok.decode([i]).lower())
        if t in SECTION_CUES: ids.append(i)
    SECTION_CUE_IDS = ids
    return len(ids)


def compute_idf(docs, vocab):
    df = np.zeros(vocab, dtype=np.float64)
    for d in docs:
        df[np.unique(d["ids"].astype(np.int64))] += 1
    return np.log((1 + len(docs)) / (1 + df)).astype(np.float32) + 1.0


# ----------------------------------------------------------------------------------------------------------------------
# 2. Examples and batches
# ----------------------------------------------------------------------------------------------------------------------
def make_prompt(d, C, idf):
    return np.concatenate([select_doc(d, C["DOC_BUDGET"], C, idf), np.asarray(C["SEP_IDS"], dtype=np.uint16)]).astype(np.int64)


def collate_lm(docs, idxs, C, idf):
    """Decoder-only / LSTM: x = prompt + summary + </s>, right-padded; y = next token on summary positions, -100 elsewhere."""
    xs, ys, starts = [], [], []
    for i in idxs:
        d = docs[i]; p = make_prompt(d, C, idf)
        s = d["sum_ids"][:C["SUM_MAX"]].astype(np.int64)
        if len(d["sum_ids"]) <= C["SUM_MAX"]: s = np.concatenate([s, [EOS_ID]])      # a summary cut at SUM_MAX gets no end token (the model must not learn to stop there)
        x = np.concatenate([p, s]); y = np.full(len(x), -100, dtype=np.int64)
        y[len(p) - 1:len(x) - 1] = s
        xs.append(x); ys.append(y); starts.append(len(p) - 1)
    T = max(len(x) for x in xs)
    X = torch.full((len(xs), T), PAD_ID, dtype=torch.long); Y = torch.full((len(xs), T), -100, dtype=torch.long); M = torch.zeros((len(xs), T), dtype=torch.bool)
    for r, (x, y) in enumerate(zip(xs, ys)):
        X[r, :len(x)] = torch.from_numpy(x); Y[r, :len(y)] = torch.from_numpy(y); M[r, :len(x)] = True
    return dict(x=X, y=Y, mask=M, starts=starts)


def collate_s2s(docs, idxs, C, idf):
    """GDN encoder-decoder: encoder = prompt (left-padded), decoder input = [last prompt token] + summary, target = summary + </s>."""
    ps, ds = [], []
    for i in idxs:
        d = docs[i]; p = make_prompt(d, C, idf)[-C["ENC_MAX_TOK"]:]
        s = d["sum_ids"][:C["SUM_MAX"]].astype(np.int64)
        if len(d["sum_ids"]) <= C["SUM_MAX"]: ds.append((np.concatenate([[p[-1]], s]), np.concatenate([s, [EOS_ID]])))
        else: ds.append((np.concatenate([[p[-1]], s[:-1]]), s))                      # cut summary: no end token
        ps.append(p)
    S = max(len(p) for p in ps); T = max(len(a) for a, _ in ds); B = len(ps)
    E = torch.full((B, S), PAD_ID, dtype=torch.long); EM = torch.zeros((B, S), dtype=torch.bool)
    DI = torch.full((B, T), PAD_ID, dtype=torch.long); DY = torch.full((B, T), -100, dtype=torch.long)
    EP = torch.zeros((B, S), dtype=torch.long); DP = torch.zeros((B, T), dtype=torch.long)
    for r, (p, (a, b)) in enumerate(zip(ps, ds)):
        pad = S - len(p)
        E[r, pad:] = torch.from_numpy(p); EM[r, pad:] = True; EP[r, pad:] = torch.arange(len(p))
        DI[r, :len(a)] = torch.from_numpy(a); DY[r, :len(b)] = torch.from_numpy(b); DP[r] = len(p) - 1 + torch.arange(T)
    return dict(enc=E, enc_mask=EM if not bool(EM.all()) else None, enc_pos=EP, dec=DI, y=DY, dec_pos=DP)


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 3. Model wrappers (expose what apply_technique needs: backbone with layer_list / adapter_sites / enable_prefix ...)
# ----------------------------------------------------------------------------------------------------------------------
class LSTMWhole(nn.Module):
    """The Phase-1 LSTM LM as a 'backbone' (its own output layer included: the LM head is part of the model)."""
    def __init__(self, lm):
        super().__init__()
        self.lm = lm; self.original_param_count = sum(p.numel() for p in lm.parameters()); self.d_model = lm.lstm.hidden_size
    def layer_list(self): return [self.lm.lstm]
    def final_modules_params(self): return list(self.lm.proj.parameters())
    def embedding_weight(self): return self.lm.emb.weight
    def adapter_sites(self, kind): raise NotImplementedError("adapters are not wired for the LSTM summariser")
    def enable_prefix(self, P, hidden=0): raise NotImplementedError("prefix tuning does not apply to an LSTM")


class GDNWhole(nn.Module):
    """The Phase-1 GDN encoder-decoder, encoder AND decoder, as a 'backbone'."""
    def __init__(self, lm):
        super().__init__()
        self.lm = lm; self.original_param_count = sum(p.numel() for p in lm.parameters()); self.d_model = lm.token_emb.embedding_dim
    def layer_list(self): return list(self.lm.enc_blocks) + list(self.lm.dec_blocks)
    def final_modules_params(self): return list(self.lm.enc_norm.parameters()) + list(self.lm.dec_norm.parameters())
    def embedding_weight(self): return self.lm.token_emb.weight
    def adapter_sites(self, kind):
        d = self.d_model
        if kind == "houlsby":
            return [(m, d) for b in self.lm.enc_blocks for m in (b.fwd, b.bwd, b.ff)] + [(m, d) for b in self.lm.dec_blocks for m in (b.self_attn, b.cross_attn, b.ff)]
        if kind == "pfeiffer":
            return [(b.ff, d) for b in self.lm.enc_blocks] + [(b.ff, d) for b in self.lm.dec_blocks]
        raise ValueError(kind)
    def enable_prefix(self, P, hidden=0): raise NotImplementedError("prefix tuning is not wired for the GDN encoder-decoder")


class SummModel(nn.Module):
    """kind: decoder (GQA / MoE: DecoderBackbone) | mor (ForwardBackbone) | lstm | gdn. logits(batch) -> (B, T, V)."""
    def __init__(self, kind, lm, C):
        super().__init__()
        self.kind = kind; self.soft_prompt = None
        if kind in ("decoder", "mor"):
            fc = lm.fc_out
            self.lm_head = fc[-1] if isinstance(fc, nn.Sequential) else fc
            self.backbone = DecoderBackbone(lm, bidirectional=False) if kind == "decoder" else ForwardBackbone(lm)
        elif kind == "lstm":
            self.lm_head = None; self.backbone = LSTMWhole(lm)
        elif kind == "gdn":
            self.lm_head = None; self.backbone = GDNWhole(lm)
        else:
            raise ValueError(kind)
        self.ckpt = C.get("CKPT", True)

    def head_parameters(self):
        return []                                   # no task head: the Phase-1 LM head generates the summary

    def logits(self, b):
        if self.kind in ("decoder", "mor"):
            h = self.backbone.features(b["x"], b["mask"], self.soft_prompt)
            return self.lm_head(h)
        if self.kind == "lstm":
            return self.backbone.lm(b["x"])
        lm = self.backbone.lm
        return lm(b["enc"], b["dec"], b["enc_mask"], b["enc_pos"], b["dec_pos"])

    def loss(self, b, C, teacher_logits=None):
        lg = self.logits(b).float()
        y = b["y"]
        ce = F.cross_entropy(lg.reshape(-1, lg.shape[-1]), y.reshape(-1), ignore_index=-100)
        if teacher_logits is None:
            return ce, lg
        T = C["KD_TEMPERATURE"]; m = (y != -100)
        s_lp = F.log_softmax(lg[m] / T, -1); t_p = F.softmax(teacher_logits[m].float() / T, -1)
        kd = F.kl_div(s_lp, t_p, reduction="batchmean") * T * T
        return (1 - C["KD_ALPHA"]) * ce + C["KD_ALPHA"] * kd, lg


def kd_topk_loss(lg, y, t_idx, t_lp, T):
    """Word-level (logit) KD from a teacher's stored top-k log-probabilities at the summary positions.
    lg [B,L,V] student logits, y [B,L] targets (-100 = no loss), t_idx / t_lp [n_pos, k] for the positions with y != -100 (row-major order).
    Both distributions are restricted to the teacher's top-k tokens and renormalised there; returns T^2 * KL(teacher || student)."""
    m = y != -100
    s = lg[m].float() / T                                            # [n_pos, V]
    s_lp = torch.log_softmax(s, -1).gather(1, t_idx)                  # student log-probs of the teacher's top-k tokens
    s_lp = s_lp - torch.logsumexp(s_lp, -1, keepdim=True)
    t_lp = torch.log_softmax(t_lp.float() / T, -1)                    # teacher, renormalised over its top-k
    return (t_lp.exp() * (t_lp - s_lp)).sum(-1).mean() * T * T


def summary_positions_logits(model, b):
    """Logits of the summary positions only, as a flat list per example (for distillation across different prompt lengths)."""
    lg = model.logits(b); m = b["y"] != -100
    return lg, m


# ----------------------------------------------------------------------------------------------------------------------
# 4. Generation (greedy or top-p sampling; no-repeat n-gram blocking; min / max length; repetition penalty)
# ----------------------------------------------------------------------------------------------------------------------
CACHE_TECHNIQUES = {"frozen", "full_ft", "partial", "lora", "rslora", "lorma", "adalora", "qlora", "ia3", "bitfit", "adapters"}


def _ban_ngrams(out_rows, n, logits):
    if n <= 0:
        return
    for r, row in enumerate(out_rows):
        if len(row) >= n - 1:
            prefix = tuple(row[len(row) - (n - 1):]) if n > 1 else ()
            banned = [row[i + n - 1] for i in range(len(row) - n + 1) if tuple(row[i:i + n - 1]) == prefix]
            if banned:
                logits[r, banned] = float("-inf")


def _choose(logits, out_rows, t, G, gen):
    logits = logits.float()
    if G.get("REP_PENALTY", 1.0) != 1.0:
        for r, row in enumerate(out_rows):
            if row:
                u = torch.tensor(sorted(set(row)), device=logits.device)
                v = logits[r, u]; logits[r, u] = torch.where(v > 0, v / G["REP_PENALTY"], v * G["REP_PENALTY"])
    _ban_ngrams(out_rows, G.get("NO_REPEAT_NGRAM", 0), logits)
    logits[:, PAD_ID] = float("-inf")
    if t < G["MIN_NEW"]:
        logits[:, EOS_ID] = float("-inf")
    if G.get("STRATEGY", "greedy") == "sample":
        p = torch.softmax(logits / max(G.get("TEMPERATURE", 1.0), 1e-5), -1)
        sp, si = torch.sort(p, descending=True); cum = sp.cumsum(-1)
        sp[(cum - sp) > G.get("TOP_P", 0.9)] = 0
        k = torch.multinomial(sp / sp.sum(-1, keepdim=True), 1, generator=gen).squeeze(-1)
        return si.gather(1, k.unsqueeze(1)).squeeze(1)
    return logits.argmax(-1)


@torch.no_grad()
def generate(model, prompts, C, G, device, gen=None):
    """prompts: list of 1-D int arrays (already ending with the separator). Returns a list of token-id lists (without </s>)."""
    model.eval()
    B = len(prompts); out = [[] for _ in range(B)]; done = torch.zeros(B, dtype=torch.bool, device=device)
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    kind = model.kind; cached = G.get("USE_CACHE", True) and C["TECHNIQUE"] in CACHE_TECHNIQUES and kind in ("decoder", "lstm", "gdn")
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        if kind == "lstm":                          # one document at a time (left padding would change the LSTM state)
            res = []
            for p in prompts:
                lm = model.backbone.lm
                lg, st = lm.step(torch.as_tensor(p, device=device).unsqueeze(0))
                row = []
                for t in range(G["MAX_NEW"]):
                    tok = _choose(lg, [row], t, G, gen)
                    if int(tok) == EOS_ID: break
                    row.append(int(tok)); lg, st = lm.step(tok.view(1, 1), st)
                res.append(row)
            return res
        if kind == "gdn":
            lm = model.backbone.lm; S = max(len(p) for p in prompts); pads = [S - len(p) for p in prompts]
            E = torch.full((B, S), PAD_ID, dtype=torch.long, device=device)
            for r, p in enumerate(prompts): E[r, pads[r]:] = torch.as_tensor(p, device=device)
            cache = EncDecCache(lm, B, device, torch.float16 if use_amp else torch.float32)
            lg = lm.start_generation(E, pads, cache)
            step = lambda tok: lm.decode_step(tok, cache)
        elif cached:                                # GQA / MoE decoders: KV cache (prefill once, then one token per step)
            lm = model.backbone.lm; fn = model.backbone.final_norm; head = model.lm_head
            P = max(len(p) for p in prompts); pads = [P - len(p) for p in prompts]
            X = torch.full((B, P), PAD_ID, dtype=torch.long, device=device)
            for r, p in enumerate(prompts): X[r, pads[r]:] = torch.as_tensor(p, device=device)
            a0 = lm.blocks[0].attn
            cache = KVCache(len(lm.blocks), B, a0.n_kv_heads, P + G["MAX_NEW"] + 1, a0.head_dim, device, torch.float16 if use_amp else torch.float32)
            pad_t = torch.tensor(pads, dtype=torch.long, device=device); hp = any(x > 0 for x in pads)
            lg = head(fn(lm.prefill(X, pads, cache)))
            step = lambda tok: head(fn(lm.decode_step(tok, pad_t, hp, cache)))
        else:                                       # full recompute (prefix / prompt tuning, MoR): every step re-reads the whole sequence
            seqs = [list(map(int, p)) for p in prompts]
            def run():
                T = max(len(s) for s in seqs)
                X = torch.full((B, T), PAD_ID, dtype=torch.long, device=device); M = torch.zeros((B, T), dtype=torch.bool, device=device)
                for r, s in enumerate(seqs): X[r, :len(s)] = torch.as_tensor(s, device=device); M[r, :len(s)] = True
                h = model.logits(dict(x=X, mask=M))
                return h[torch.arange(B, device=device), torch.tensor([len(s) - 1 for s in seqs], device=device)]
            lg = run()
            def step(tok):
                for r in range(B): seqs[r].append(int(tok[r]))
                return run()
        for t in range(G["MAX_NEW"]):
            tok = _choose(lg, out, t, G, gen)
            tok = torch.where(done, torch.full_like(tok, EOS_ID), tok)
            for r in range(B):
                if not bool(done[r]) and int(tok[r]) != EOS_ID: out[r].append(int(tok[r]))
            done |= tok == EOS_ID
            if bool(done.all()) or t == G["MAX_NEW"] - 1: break
            lg = step(tok)
    return out


# ----------------------------------------------------------------------------------------------------------------------
# 5. Scores: BERTScore (roberta-large, as in Phase 1), ROUGE, length, repetition, novelty, number faithfulness
# ----------------------------------------------------------------------------------------------------------------------
def _words(t): return re.findall(r"\w+", t.lower())
NUM = re.compile(r"\b\d+[\d./()-]*\b")


def summary_scores(hyps, refs, docs, device, with_bertscore=True):
    """BERTScore F1, ROUGE-1 and ROUGE-L computed exactly as the course evaluator (evaluate_phase6.py) does, plus ROUGE-2,
    length, repetition, novelty and number statistics."""
    from rouge_score import rouge_scorer, tokenizers
    hyps = [h.strip() if h.strip() else "." for h in hyps]; refs = [r.strip() for r in refs]
    rs = rouge_scorer.RougeScorer(["rouge1", "rouge2"], use_stemmer=True); tok = tokenizers.DefaultTokenizer(use_stemmer=True)
    r1, r2, rl = [], [], []
    for h, ref in zip(hyps, refs):
        s = rs.score(ref, h); r1.append(s["rouge1"].fmeasure); r2.append(s["rouge2"].fmeasure)
        pt, rt = tok.tokenize(h), tok.tokenize(ref)
        if not pt and not rt: x = 1.0
        elif not pt or not rt: x = 0.0
        else:
            z = lcs(pt, rt); pp, rr = z / len(pt), z / len(rt); x = 0.0 if pp + rr == 0 else 2 * pp * rr / (pp + rr)
        rl.append(x)
    out = dict(rouge1=float(np.mean(r1)), rouge2=float(np.mean(r2)), rougeL=float(np.mean(rl)), n=len(hyps))
    if with_bertscore:
        import bert_score
        P_, R_, F_ = bert_score.score(hyps, refs, lang="en", batch_size=8, verbose=False, device=str(device))
        out.update(bertscore_f1=float(F_.mean()), bertscore_p=float(P_.mean()), bertscore_r=float(R_.mean()), bertscore_f1_each=F_.tolist())
    rep, nov, hal, cov = [], [], [], []
    for h, ref, d in zip(hyps, refs, docs):
        w = _words(h); g = list(zip(w, w[1:], w[2:], w[3:]))
        rep.append(1 - len(set(g)) / len(g) if g else 0.0)
        dw = _words(d); dg = set(zip(dw, dw[1:], dw[2:]))
        g3 = list(zip(w, w[1:], w[2:])); nov.append(np.mean([x not in dg for x in g3]) if g3 else 0.0)
        hn, dn, rn = set(NUM.findall(h)), set(NUM.findall(d)), set(NUM.findall(ref))
        if hn: hal.append(len(hn - dn) / len(hn))
        if rn: cov.append(len(rn & hn) / len(rn))
    out.update(gen_words=float(np.mean([len(h.split()) for h in hyps])), ref_words=float(np.mean([len(x.split()) for x in refs])),
               repeated_4gram=float(np.mean(rep)), novel_3gram=float(np.mean(nov)), numbers_not_in_doc=float(np.mean(hal)) if hal else None,
               ref_numbers_covered=float(np.mean(cov)) if cov else None, empty=int(sum(1 for h in hyps if not h.strip())))
    return out


# ----------------------------------------------------------------------------------------------------------------------
# 6. Training
# ----------------------------------------------------------------------------------------------------------------------
def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup: return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith((".tau", ".tau_ready", ".ada_mask")):
            st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


@torch.no_grad()
def dev_loss(model, docs, C, idf, device):
    model.eval(); use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    tot, cnt = 0.0, 0; col = collate_s2s if model.kind == "gdn" else collate_lm
    for a in range(0, len(docs), C["EVAL_BATCH"]):
        b = to_device(col(docs, list(range(a, min(a + C["EVAL_BATCH"], len(docs)))), C, idf), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model.logits(b).float()
        y = b["y"]; m = y != -100
        tot += F.cross_entropy(lg[m], y[m], reduction="sum").item(); cnt += int(m.sum())
    return tot / max(cnt, 1)


# ---------------------------------------------------------------------------------------------- optimizers (OPTIMIZER setting)
class Lion(torch.optim.Optimizer):
    """Lion (Chen et al. 2023): sign of an interpolated momentum; decoupled weight decay."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for g in self.param_groups:
            b1, b2 = g["betas"]
            for p in g["params"]:
                if p.grad is None: continue
                st = self.state[p]
                if "m" not in st: st["m"] = torch.zeros_like(p)
                m = st["m"]; p.mul_(1 - g["lr"] * g["weight_decay"])
                p.add_(torch.sign(m * b1 + p.grad * (1 - b1)), alpha=-g["lr"]); m.mul_(b2).add_(p.grad, alpha=1 - b2)


def _newton_schulz5(G, steps=5):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float(); tr = X.shape[0] > X.shape[1]
    if tr: X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T; X = a * X + (b * A + c * A @ A) @ X
    return X.T if tr else X


class MuonAdamW(torch.optim.Optimizer):
    """Muon (momentum + Newton-Schulz orthogonalised update) for 2-D weight matrices, AdamW for every other parameter.
    (The Phase-1 MoE was pre-trained with Muon.)"""
    def __init__(self, params, lr=0.02, momentum=0.95, adamw_lr=3e-4, betas=(0.9, 0.98), weight_decay=0.0):
        mu = [p for p in params if p.ndim == 2 and min(p.shape) > 1]; ad = [p for p in params if not (p.ndim == 2 and min(p.shape) > 1)]
        groups = [dict(params=mu, muon=True, lr=lr)] + ([dict(params=ad, muon=False, lr=adamw_lr)] if ad else [])
        super().__init__(groups, dict(momentum=momentum, betas=betas, weight_decay=weight_decay, eps=1e-8))
    @torch.no_grad()
    def step(self, closure=None):
        for g in self.param_groups:
            for p in g["params"]:
                if p.grad is None: continue
                st = self.state[p]
                if g["muon"]:
                    if "m" not in st: st["m"] = torch.zeros_like(p)
                    m = st["m"]; m.mul_(g["momentum"]).add_(p.grad)
                    u = _newton_schulz5(p.grad + g["momentum"] * m).to(p.dtype) * max(1.0, p.shape[0] / p.shape[1]) ** 0.5
                    p.mul_(1 - g["lr"] * g["weight_decay"]); p.add_(u, alpha=-g["lr"])
                else:
                    if "t" not in st: st.update(t=0, m=torch.zeros_like(p), v=torch.zeros_like(p))
                    st["t"] += 1; b1, b2 = g["betas"]; st["m"].mul_(b1).add_(p.grad, alpha=1 - b1); st["v"].mul_(b2).addcmul_(p.grad, p.grad, value=1 - b2)
                    mh = st["m"] / (1 - b1 ** st["t"]); vh = st["v"] / (1 - b2 ** st["t"])
                    p.mul_(1 - g["lr"] * g["weight_decay"]); p.addcdiv_(mh, vh.sqrt().add_(g["eps"]), value=-g["lr"])


def make_optimizer(model, params, C):
    o = C.get("OPTIMIZER", "adamw"); lr, wd = C["LR"], C["WEIGHT_DECAY"]
    if o == "adamw": return torch.optim.AdamW(params, lr=lr, weight_decay=wd, betas=(0.9, 0.98))
    if o == "adam": return torch.optim.Adam(params, lr=lr, betas=(0.9, 0.98))
    if o == "sgd": return torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=0.0)
    if o == "rmsprop": return torch.optim.RMSprop(params, lr=lr, alpha=0.99, weight_decay=0.0)
    if o == "adafactor": return torch.optim.Adafactor(params, lr=lr, weight_decay=wd)
    if o == "lion": return Lion(params, lr=lr, weight_decay=wd * 10)
    if o == "muon": return MuonAdamW(params, lr=lr, adamw_lr=C.get("ADAMW_LR_WITH_MUON", 3e-4), weight_decay=0.0)
    raise ValueError(o)


def lm_window_loss(model, docs, L, device, use_amp, n=100):
    """Next-token loss (nats) on the first L+1 tokens of up to n documents (the domain-adaptation check)."""
    model.eval(); tot, cnt = 0.0, 0
    with torch.no_grad():
        for d in docs[:n]:
            w = torch.as_tensor(d["ids"][:L + 1].astype(np.int64), device=device).unsqueeze(0)
            b = dict(x=w[:, :-1], mask=torch.ones_like(w[:, :-1], dtype=torch.bool), y=w[:, 1:])
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                lg = model.logits(b)
            tot += F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), b["y"].reshape(-1), reduction="sum").item(); cnt += b["y"].numel()
    return tot / max(1, cnt)


def dapt(model, docs, dev_docs, C, device, log=print):
    """Domain / task-adaptive pre-training before summarisation fine-tuning: DAPT_EPOCHS passes of plain next-token training on
    random CONTEXT-token windows of the TRAINING judgments (no summaries), with the same trainable parameters as the technique."""
    assert model.kind in ("decoder", "mor"), "DAPT is wired for the decoder-only models"
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"; L = C["CONTEXT"]
    before = lm_window_loss(model, dev_docs, L, device, use_amp)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=C.get("DAPT_LR") or C["LR"], weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    total = math.ceil(len(docs) * C["DAPT_EPOCHS"] / C["ACCUM"]); sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(0.05 * total)))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp); rng = random.Random(C["SEED"] + 99); t0 = time.time(); pending, run, nb = 0, 0.0, 0
    for ep in range(C["DAPT_EPOCHS"]):
        model.train(); order = list(range(len(docs))); rng.shuffle(order); opt.zero_grad(set_to_none=True)
        for j, i in enumerate(order):
            ids = docs[i]["ids"]; s0 = rng.randint(0, len(ids) - L - 1) if len(ids) > L + 1 else 0
            w = torch.as_tensor(ids[s0:s0 + L + 1].astype(np.int64), device=device).unsqueeze(0)
            b = dict(x=w[:, :-1], mask=torch.ones_like(w[:, :-1], dtype=torch.bool), y=w[:, 1:])
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss, _ = model.loss(b, C)
            if not torch.isfinite(loss): continue
            scaler.scale(loss / C["ACCUM"]).backward(); pending += 1; run += loss.item(); nb += 1
            if pending >= C["ACCUM"] or j == len(order) - 1:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, C["GRAD_CLIP"]); scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True); pending = 0
        log(f"  DAPT epoch {ep + 1}: train LM loss {run / max(1, nb):.4f} [{(time.time() - t0) / 60:.1f} min]"); run, nb = 0.0, 0
    after = lm_window_loss(model, dev_docs, L, device, use_amp)
    log(f"  DAPT: next-token loss on dev judgments {before:.4f} -> {after:.4f} nats ({(time.time() - t0) / 60:.1f} min)")
    return dict(dapt_lm_loss_before=round(before, 4), dapt_lm_loss_after=round(after, 4), dapt_minutes=round((time.time() - t0) / 60, 1))


def train_summ(model, train_docs, dev_docs, C, device, idf, log=print, teacher=None, ckpt_save=None, ckpt_load=None, kd_topk=None):
    """EPOCHS epochs of EPOCH_DOCS documents each (a reshuffled stream). After every epoch: teacher-forced dev loss on the dev
    documents; the best epoch's trainable state is kept. Optional teacher (a SummModel with the same input format) for KD."""
    model.to(device); torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    N = len(train_docs); per_epoch = min(C["EPOCH_DOCS"] or N, N); epochs = C["EPOCHS"]
    params = [p for p in model.parameters() if p.requires_grad]
    opt = make_optimizer(model, params, C)
    steps_per_epoch = math.ceil(per_epoch / (C["BATCH"] * C["ACCUM"])); total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
    col = collate_s2s if model.kind == "gdn" else collate_lm
    rng = random.Random(C["SEED"] * 1000 + 7); stream = []
    def take(k):
        nonlocal stream
        while len(stream) < k:
            p = list(range(N)); rng.shuffle(p); stream += p
        o, stream = stream[:k], stream[k:]
        return o
    hist, best, best_state, best_ep = [], float("inf"), None, 0
    t0, tokens, gstep, start_ep = time.time(), 0, 0, 1
    ck = ckpt_load() if ckpt_load else None
    if ck is not None:
        load_trainable_state(model, ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        rng.setstate(ck["rng"]); stream = ck["stream"]; torch.set_rng_state(ck["torch_rng"])
        hist, best, best_state, best_ep = ck["hist"], ck["best"], ck["best_state"], ck["best_ep"]
        tokens, gstep = ck["tokens"], ck["gstep"]; t0 = time.time() - ck["elapsed"]; start_ep = ck["epoch"] + 1
        log(f"  resumed after epoch {ck['epoch']} (best so far: epoch {best_ep}, dev loss {best:.4f})")
    log(f"training: {N} documents, {per_epoch} per epoch x {epochs} epochs, {steps_per_epoch} optimizer steps per epoch, {sum(p.numel() for p in params):,} trainable parameters" + (", KD from a teacher" if teacher is not None else ""))
    for ep in range(start_ep, epochs + 1):
        model.train(); order = take(per_epoch); run_loss, nb, pending, skipped, kd_mismatch = 0.0, 0, 0, 0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            if C.get("TIME_BUDGET_MIN") and pending == 0 and (time.time() - t0) / 60 > C["TIME_BUDGET_MIN"]:
                log(f"  time budget of {C['TIME_BUDGET_MIN']} min reached inside epoch {ep} after {bi} of {len(order)} documents"); break
            b = to_device(col(train_docs, order[bi:bi + C["BATCH"]], C, idf), device)
            tl = None
            if teacher is not None:
                with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    tl = teacher.logits(b)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss, lg_s = model.loss(b, C, tl)
            if kd_topk is not None:                  # word-level KD from stored teacher top-k log-probs (documents without them: CE only)
                ids_b = [train_docs[i]["id"] for i in order[bi:bi + C["BATCH"]]]
                if all(i in kd_topk for i in ids_b):
                    ti = torch.cat([torch.from_numpy(kd_topk[i][0].astype(np.int64)) for i in ids_b]).to(device)
                    tp = torch.cat([torch.from_numpy(kd_topk[i][1].astype(np.float32)) for i in ids_b]).to(device)
                    if ti.shape[0] == int((b["y"] != -100).sum()):
                        loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd_topk_loss(lg_s, b["y"], ti, tp, C["KD_TEMPERATURE"])
                    else:
                        kd_mismatch += 1
            del lg_s
            if hasattr(model, "extra_loss"):
                loss = loss + model.extra_loss()
            if not torch.isfinite(loss):
                skipped += 1; continue
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1; pending += 1; run_loss += loss.item(); tokens += int((b["y"] != -100).sum())
            if pending >= C["ACCUM"] or (bi + C["BATCH"] >= len(order) and pending > 0):
                scaler.unscale_(opt)
                if hasattr(model, "on_before_step"): model.on_before_step(gstep, total)
                torch.nn.utils.clip_grad_norm_(params, C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); gstep += 1
                opt.zero_grad(set_to_none=True); pending = 0
        t_e = time.time(); dl = dev_loss(model, dev_docs, C, idf, device)
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 4), dev_loss=round(dl, 4), dev_ppl=round(math.exp(min(dl, 20)), 2), train_seconds=round(time.time() - t0, 1), eval_seconds=round(time.time() - t_e, 1))
        if skipped: row["skipped_nonfinite"] = skipped
        if kd_mismatch: row["kd_length_mismatch"] = kd_mismatch
        if dl < best:
            best, best_ep, best_state = dl, ep, trainable_state(model)
        hist.append(row); log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if ckpt_save and ep < epochs:
            ckpt_save(dict(epoch=ep, model=trainable_state(model), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(), rng=rng.getstate(), stream=list(stream),
                           torch_rng=torch.get_rng_state(), hist=hist, best=best, best_state=best_state, best_ep=best_ep, tokens=tokens, gstep=gstep, elapsed=time.time() - t0))
        if C.get("TIME_BUDGET_MIN") and (time.time() - t0) / 60 > C["TIME_BUDGET_MIN"]:
            log(f"  time budget reached after epoch {ep}"); break
    train_secs = time.time() - t0 - sum(r["eval_seconds"] for r in hist)
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_dev_loss=best, best_state=best_state, train_seconds=round(train_secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(train_secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 7. Submission
# ----------------------------------------------------------------------------------------------------------------------
def write_submission(ids, summaries, out_dir, name, expected_ids):
    """JSON list [{"id": ..., "summary": "..."}], exactly one per test document, in a zip with prediction.json at its root."""
    assert len(ids) == len(set(ids)) == len(expected_ids) and set(ids) == set(expected_ids), "ids differ from test.json"
    sub = [dict(id=i, summary=(s.strip() if s.strip() else "summary")) for i, s in zip(ids, summaries)]
    os.makedirs(out_dir, exist_ok=True)
    jpath = os.path.join(out_dir, f"{name}_prediction.json"); json.dump(sub, open(jpath, "w", encoding="utf-8"), ensure_ascii=False)
    chk = json.load(open(jpath, encoding="utf-8"))
    assert isinstance(chk, list) and all(isinstance(r["id"], type(expected_ids[0])) and isinstance(r["summary"], str) and r["summary"] for r in chk)
    zpath = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    return jpath, zpath


# ===================== The Phase-6 evaluator (course file evaluate_phase6.py; load / lcs / read_pred and the three scores unchanged, main() became a function) =====================
from rouge_score import tokenizers, rouge_scorer
import bert_score

def load(path):
    with open(path,encoding='utf-8') as f: return json.load(f)

def lcs(a,b):
    if len(a)>len(b): a,b=b,a
    masks={}
    for i,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<i)
    s=0
    for t in b:
        m=masks.get(t,0); x=m|s; y=(s<<1)|1; s=x & ~(x-y)
    return s.bit_count() if hasattr(int,'bit_count') else bin(s).count('1')

def read_pred(x):
    if isinstance(x,dict): out=x
    elif isinstance(x,list):
        out={}
        for r in x:
            if not isinstance(r,dict) or 'id' not in r or 'summary' not in r: raise ValueError('Each prediction item must contain id and summary.')
            if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
            out[r['id']]=r['summary']
    else: raise ValueError('Prediction must be a JSON object or list of objects.')
    for k,v in out.items():
        if not isinstance(k,str) or not isinstance(v,str) or not v.strip(): raise ValueError(f'Invalid summary for ID {k!r}.')
    return out

def official_eval(truth_path, prediction_path, verbose=True):
    t=load(truth_path); p=read_pred(load(prediction_path))
    if not isinstance(t,dict) or not t: raise ValueError('Truth must be a non-empty JSON object.')
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    preds=[p[k].strip() for k in t]; refs=[t[k].strip() for k in t]
    scorer=rouge_scorer.RougeScorer(['rouge1'],use_stemmer=True); tok=tokenizers.DefaultTokenizer(use_stemmer=True)
    r1=rl=0.0
    for pr,rf in zip(preds,refs):
        s=scorer.score(rf,pr); r1+=s['rouge1'].fmeasure
        pt=tok.tokenize(pr); rt=tok.tokenize(rf)
        if not pt and not rt: x=1.0
        elif not pt or not rt: x=0.0
        else:
            z=lcs(pt,rt); pp=z/len(pt); rr=z/len(rt); x=0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)
        rl+=x
    n=len(t); r1/=n; rl/=n
    try:
        import torch; dev='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: dev=None
    _,_,f=bert_score.score(preds,refs,lang='en',batch_size=8,verbose=False,**({'device':dev} if dev else {}))
    res=dict(documents=n, bertscore_f1=float(f.mean().item()), rouge1=r1, rougeL=rl)
    if verbose: print(f"  [official evaluator] Documents: {n}  BERTScore F1: {res['bertscore_f1']:.6f}  ROUGE-1: {res['rouge1']:.6f}  ROUGE-L: {res['rougeL']:.6f}")
    return res

def check_submission(pred_path, test_ids):
    p=read_pred(load(pred_path))
    if set(p)!=set(test_ids) or len(p)!=len(test_ids): raise ValueError(f"ID mismatch with test.json: missing={len(set(test_ids)-set(p))}, extra={len(set(p)-set(test_ids))}")
    return True

## 7. Adaptation techniques

In [7]:
# ===================== Phase 3 adaptation techniques: frozen, full FT, LoRA, rsLoRA, LoRMA, AdaLoRA, QLoRA, IA3, prompt, prefix =====================
# All of them leave the Phase-1 model code untouched: they wrap/replace nn.Linear modules inside the backbone, or add small trainable tensors.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------------------------------------------------
# LoRA / rsLoRA / LoRMA
# ----------------------------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2022): y = W0 x + b + s * B A x,  A (r, in) random, B (out, r) zeros -> starts exactly at W0.
    s = alpha / r (LoRA)  or  alpha / sqrt(r) (rank-stabilised LoRA, Kalajdzievski 2023: keeps the update size stable when r grows)."""

    def __init__(self, base, r, alpha, dropout=0.0, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.r = r
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


_ROT_IDX = {}


def _row_rotation_index(d, device):
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device).unsqueeze(1)
        j = torch.arange(d, device=device).unsqueeze(0)
        _ROT_IDX[key] = (j - i) % d                      # Mpi[i, j] = M[i, (j - i) mod d]  (row i rotated right by i)
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel, Modi, ACL Findings 2025): a MULTIPLICATIVE low-rank update of a frozen W0 (out x in).
      "plus" (LoRMA+):  h = ((alpha/r) B A + I) W0 x + b,   A (r, out) random, B (out, r) zeros -> starts at W0. Computed as
                        y0 = W0 x;  y0 + (alpha/r) B (A y0), so the out x out matrix is never formed.
      "pi"   (LoRMA_pi): h = I_pi(B A) W0 x + b, I_pi rotates row i of BA cyclically by i (rank inflation). Init so that I_pi(BA) = I."""

    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        self.base, self.variant, self.r, self.scale = base, variant, r, alpha / r
        d = base.out_features
        if variant == "plus":
            self.A = nn.Parameter(torch.empty(r, d))
            self.B = nn.Parameter(torch.zeros(d, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        elif variant == "pi":
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.randn(d, r) * (1.0 / math.sqrt(d)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
        else:
            raise ValueError(variant)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        if self.variant == "plus":
            y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        else:
            M = self.B @ self.A
            Mpi = torch.gather(M, 1, _row_rotation_index(M.shape[0], M.device))
            y = F.linear(y0, Mpi.to(y0.dtype))
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


# ----------------------------------------------------------------------------------------------------------------------
# AdaLoRA (Zhang et al. 2023), simplified
# ----------------------------------------------------------------------------------------------------------------------
class AdaLoRALinear(nn.Module):
    """y = W0 x + b + s * P diag(lam * mask) Q x   (an SVD-shaped update: P (out, r) and Q (r, in) start orthonormal, lam (r) starts at 0).
    The controller below prunes whole (P_i, lam_i, Q_i) triplets by importance during training, moving the rank budget to the layers that
    need it. Simplification: a pruned triplet stays pruned."""

    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        out, inn = base.out_features, base.in_features
        P = torch.linalg.qr(torch.randn(out, r))[0]                      # orthonormal columns
        Q = torch.linalg.qr(torch.randn(inn, r))[0].t().contiguous()     # orthonormal rows
        self.P, self.Q = nn.Parameter(P), nn.Parameter(Q)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("ada_mask", torch.ones(r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        z = F.linear(self.drop(x), self.Q) * (self.lam * self.ada_mask)
        return self.base(x) + self.scale * F.linear(z, self.P)

    def orth_penalty(self):
        eye = torch.eye(self.r, device=self.P.device)
        return ((self.P.t() @ self.P - eye) ** 2).sum() + ((self.Q @ self.Q.t() - eye) ** 2).sum()


class AdaLoRAController:
    """Importance score of a triplet i: s(lam_i) + mean_j s(P_ji) + mean_j s(Q_ij), with s(w) = smoothed |w * grad(w)| times its
    smoothed uncertainty (paper, eq. 8-10). Budget schedule: full budget for the first t_i steps, then a cubic decrease to the target
    budget, then fixed for the last t_f steps. Also adds the orthogonality penalty to the loss."""

    def __init__(self, modules, target_rank, ti_frac=0.1, tf_frac=0.2, delta_t=10, beta=0.85, orth_weight=0.1):
        self.mods = modules
        self.r0 = modules[0].r
        self.b0 = len(modules) * self.r0
        self.bT = int(round(len(modules) * target_rank))
        self.ti_frac, self.tf_frac, self.dt, self.beta, self.orth = ti_frac, tf_frac, delta_t, beta, orth_weight
        z = lambda p: torch.zeros_like(p.data)
        self.st = [{n: (z(getattr(m, n)), z(getattr(m, n))) for n in ("lam", "P", "Q")} for m in modules]

    def extra_loss(self):
        return self.orth * sum(m.orth_penalty() for m in self.mods) / len(self.mods)

    def budget(self, step, total):
        ti, tf = int(self.ti_frac * total), int(self.tf_frac * total)
        if step < ti:
            return self.b0
        if step > total - tf:
            return self.bT
        frac = (step - ti) / max(1, total - ti - tf)
        return int(round(self.bT + (self.b0 - self.bT) * (1 - frac) ** 3))

    @torch.no_grad()
    def on_before_step(self, step, total):
        b = self.beta
        for m, st in zip(self.mods, self.st):
            for n, (ibar, u) in list(st.items()):
                p = getattr(m, n)
                if ibar.device != p.device:                       # the statistics were created on the CPU, the model may have moved since
                    ibar, u = ibar.to(p.device), u.to(p.device); st[n] = (ibar, u)
                if p.grad is None:
                    continue
                imp = (p.data * p.grad).abs()
                ibar.mul_(b).add_(imp, alpha=1 - b)
                u.mul_((b)).add_((imp - ibar).abs(), alpha=1 - b)
        tf = int(self.tf_frac * total)
        if step % self.dt == 0 and step <= total - tf:
            scores = []
            for m, st in zip(self.mods, self.st):
                s = lambda n: st[n][0] * st[n][1]
                scores.append(s("lam") + s("P").mean(0) + s("Q").mean(1))
            allsc = torch.cat(scores)
            keep = min(max(self.budget(step, total), 1), allsc.numel())
            thr = torch.sort(allsc, descending=True).values[keep - 1]
            for m, sc in zip(self.mods, scores):
                m.ada_mask.copy_((sc >= thr).float() * m.ada_mask)       # a pruned triplet stays pruned

    def rank_report(self):
        return [int(m.ada_mask.sum().item()) for m in self.mods]


# ----------------------------------------------------------------------------------------------------------------------
# QLoRA (Dettmers et al. 2023): frozen 4-bit NormalFloat base weights (+ double quantisation of the block constants) with LoRA on top
# ----------------------------------------------------------------------------------------------------------------------
NF4_TABLE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
                          -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                          0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """A frozen Linear stored as 4-bit NF4 codes (two per byte) with one absmax constant per `block` weights; the constants are themselves
    stored in 8 bit (double quantisation). The weight is rebuilt on the fly in every forward pass (pure PyTorch: no bitsandbytes)."""

    def __init__(self, base, block=64, double_quant=True, dq_block=256):
        super().__init__()
        self.in_features, self.out_features, self.block, self.double_quant = base.in_features, base.out_features, block, double_quant
        W = base.weight.detach().float().reshape(-1)
        self.numel = W.numel()
        Wp = torch.cat([W, W.new_zeros((-self.numel) % block)])
        blocks = Wp.view(-1, block)
        absmax = blocks.abs().amax(1).clamp(min=1e-8)
        flat = (blocks / absmax[:, None]).reshape(-1)
        codes = torch.empty(flat.numel(), dtype=torch.uint8)
        for i in range(0, flat.numel(), 1 << 21):
            ch = flat[i:i + (1 << 21)]
            codes[i:i + ch.numel()] = (ch.unsqueeze(1) - NF4_TABLE).abs().argmin(1).to(torch.uint8)
        self.register_buffer("packed", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("nf4", NF4_TABLE.clone(), persistent=False)
        self.n_blocks = absmax.numel()
        if double_quant:
            mean = absmax.mean()
            am = absmax - mean
            amp = torch.cat([am, am.new_zeros((-am.numel()) % dq_block)]).view(-1, dq_block)
            sc = amp.abs().amax(1).clamp(min=1e-12)
            self.register_buffer("am_q", torch.round(amp / sc[:, None] * 127).to(torch.int8))
            self.register_buffer("am_scale", sc)
            self.register_buffer("am_mean", mean.reshape(1))
        else:
            self.register_buffer("absmax", absmax)
        self.register_buffer("bias", base.bias.detach().clone() if base.bias is not None else None)

    def _absmax(self):
        if self.double_quant:
            return (self.am_q.float() / 127 * self.am_scale[:, None]).reshape(-1)[:self.n_blocks] + self.am_mean
        return self.absmax

    @torch.no_grad()
    def dequant(self, dtype=torch.float32):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 0x0F).long()], 1).reshape(-1)
        vals = self.nf4[codes].view(-1, self.block) * self._absmax()[:, None]
        return vals.reshape(-1)[:self.numel].view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        W = self.dequant(x.dtype)
        return F.linear(x, W, None if self.bias is None else self.bias.to(x.dtype))


# ----------------------------------------------------------------------------------------------------------------------
# IA3 (Liu et al. 2022): learned per-channel rescaling vectors, initialised to 1
# ----------------------------------------------------------------------------------------------------------------------
class IA3Out(nn.Module):
    """y = l * Linear(x): rescales the OUTPUT channels (used on the key and value projections)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """y = Linear(l * x): rescales the INPUT channels (used on the feed-forward down-projection, i.e. the hidden activation)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


# ----------------------------------------------------------------------------------------------------------------------
# Applying a technique
# ----------------------------------------------------------------------------------------------------------------------
def _swap_linears(root, fn):
    """fn(qualified_name, nn.Linear) -> replacement module or None. Returns the names that were replaced."""
    done = []
    for qual, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear):
                full = f"{qual}.{cname}" if qual else cname
                new = fn(full, child)
                if new is not None:
                    setattr(mod, cname, new); done.append(full)
    return done


def _hit(full, suffixes):
    return any(full.endswith(s) for s in suffixes)


def size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6 + sum(b.numel() * b.element_size() for b in model.buffers()) / 1e6


def apply_technique(model, C, log=print):
    """Freezes/unfreezes the backbone and inserts modules according to C['TECHNIQUE']; the sentence head is always trainable."""
    tech, bb = C["TECHNIQUE"], model.backbone
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    T = C.get("PEFT_TARGETS", [])
    if tech == "frozen":
        pass                                                           # Phase-1 weights untouched: only the head learns (control)
    elif tech == "full_ft":
        for p in bb.parameters():
            p.requires_grad = True
    elif tech in ("lora", "rslora"):
        done = _swap_linears(bb, lambda n, l: LoRALinear(l, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0), tech == "rslora") if _hit(n, T) else None)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], scale=(C["ALPHA"] / math.sqrt(C["RANK"]) if tech == "rslora" else C["ALPHA"] / C["RANK"]))
    elif tech == "lorma":
        done = _swap_linears(bb, lambda n, l: LoRMALinear(l, C["RANK"], C["ALPHA"], C["LORMA_VARIANT"]) if _hit(n, T) else None)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], variant=C["LORMA_VARIANT"])
    elif tech == "adalora":
        mods = []

        def mk(n, l):
            if _hit(n, T):
                m = AdaLoRALinear(l, C["ADALORA_INIT_RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0)); mods.append(m); return m
        done = _swap_linears(bb, mk)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        ctrl = AdaLoRAController(mods, C["ADALORA_TARGET_RANK"], delta_t=C.get("ADALORA_DELTA_T", 10), orth_weight=C.get("ADALORA_ORTH", 0.1))
        model.extra_loss, model.on_before_step, model.ada_controller = ctrl.extra_loss, ctrl.on_before_step, ctrl
        info.update(targets=len(done), init_rank=C["ADALORA_INIT_RANK"], target_rank=C["ADALORA_TARGET_RANK"])
    elif tech == "qlora":
        blocks = bb.layer_list()
        n_q, n_l = [0], [0]

        def mk(n, l):
            q = NF4Linear(l, C.get("NF4_BLOCK", 64), C.get("NF4_DOUBLE_QUANT", True))
            n_q[0] += 1
            if _hit(n, T):
                n_l[0] += 1
                return LoRALinear(q, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0))
            return q
        for blk in blocks:
            _swap_linears(blk, mk)
        assert n_l[0], f"no linear layer matches PEFT_TARGETS={T}"
        info.update(quantized_linears=n_q[0], lora_targets=n_l[0], rank=C["RANK"], alpha=C["ALPHA"])
    elif tech == "ia3":
        n_out, n_in = [0], [0]

        def mk(n, l):
            if _hit(n, C["IA3_OUT"]):
                n_out[0] += 1; return IA3Out(l)
            if _hit(n, C["IA3_IN"]):
                n_in[0] += 1; return IA3In(l)
        _swap_linears(bb, mk)
        assert n_out[0] + n_in[0], f"no linear layer matches IA3_OUT={C['IA3_OUT']} / IA3_IN={C['IA3_IN']}"
        info.update(out_scaled=n_out[0], in_scaled=n_in[0])
    elif tech == "prompt":
        P = C["PROMPT_LEN"]
        emb = bb.embedding_weight()
        idx = torch.randint(100, emb.shape[0], (P,), generator=torch.Generator().manual_seed(0))
        model.soft_prompt = nn.Parameter(emb[idx].detach().clone().float())
        info.update(prompt_len=P)
    elif tech == "prefix":
        bb.enable_prefix(C["PREFIX_LEN"], 0)
        info.update(prefix_len=C["PREFIX_LEN"])
    else:
        raise ValueError(f"unknown TECHNIQUE {tech}")
    for p in model.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=orig,
                trainable_pct_of_backbone=round(100.0 * trainable / orig, 4), model_size_mb=round(size_mb(model), 1))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {orig:,}); "
        f"stored size {info['model_size_mb']} MB")
    return info

In [8]:
# ===================== Phase 4/5/6 additions to the adaptation techniques: BitFit, partial (top-layer) fine-tuning, bottleneck adapters =====================
# Wraps the Phase-3 apply_technique (LoRA, rsLoRA, LoRMA, AdaLoRA, QLoRA, IA3, prompt, prefix, full_ft, frozen are unchanged).
_p3_apply_technique = apply_technique


class BottleneckAdapter(nn.Module):
    """Houlsby/Pfeiffer-style adapter: y + W_up act(W_down y); W_up starts at zero so the model starts exactly at the Phase-1 function."""

    def __init__(self, d, r):
        super().__init__()
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, y):
        return y + self.up(F.gelu(self.down(y)))


def _attach_adapter(m, d, r):
    ad = BottleneckAdapter(d, r)
    m.add_module("adapter", ad)
    orig = m.forward
    def fwd(*a, **k):
        return m.adapter(orig(*a, **k))
    m.forward = fwd
    if hasattr(m, "decode_step"):                       # cached generation calls decode_step, not forward: the adapter must act there too
        orig_ds = m.decode_step
        def ds(*a, **k):
            return m.adapter(orig_ds(*a, **k))
        m.decode_step = ds
    return ad


def apply_technique(model, C, log=print):
    tech, bb = C["TECHNIQUE"], model.backbone
    if tech not in ("bitfit", "partial", "adapters"):
        return _p3_apply_technique(model, C, log)
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    if tech == "bitfit":                                   # Ben Zaken et al. 2022: only the bias terms of the backbone are trained
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith("bias") or ".bias_" in name or name.endswith(("bias_ih_l0", "bias_hh_l0")):
                p.requires_grad = True; n += p.numel()
        assert n > 0, "this backbone has no bias terms"
        info.update(bias_params=n)
    elif tech == "partial":                                # layer-wise fine-tuning: the top PARTIAL_LAYERS layers + the final norm
        layers = bb.layer_list(); k = C["PARTIAL_LAYERS"]
        for l in layers[-k:]:
            for p in l.parameters():
                p.requires_grad = True
        for p in bb.final_modules_params():
            p.requires_grad = True
        info.update(unfrozen_layers=k, of_layers=len(layers))
    else:                                                  # adapters
        sites = bb.adapter_sites(C["ADAPTER_KIND"])
        for m, d in sites:
            _attach_adapter(m, d, C["ADAPTER_R"])
        info.update(adapter_kind=C["ADAPTER_KIND"], adapter_sites=len(sites), bottleneck=C["ADAPTER_R"])
    for p in model.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=orig,
                trainable_pct_of_backbone=round(100.0 * trainable / orig, 4), model_size_mb=round(size_mb(model), 1))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {orig:,}); stored size {info['model_size_mb']} MB")
    return info

## 8. Data: Phase_6_Summ, dev split, tokenisation

In [9]:
import numpy as np
P6 = "Phase_6_Summ"
UPLOAD_FAILED = []
_upload_unchecked = upload_model_file
def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    if "batch_bucket_files" in globals() and "HF_BUCKET" in globals():
        try:
            batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN); print(f"uploaded {name} to bucket {HF_BUCKET}"); return True
        except Exception as e:
            print(f"!!! UPLOAD FAILED: {name} ({type(e).__name__}: {str(e)[:200]}). The file is still in {local_path}"); UPLOAD_FAILED.append(local_path); return False
    _upload_unchecked(local_path, name); return True

def load_json(p):
    with open(p, encoding="utf-8") as f:
        x = json.load(f)
    return list(x.values()) if isinstance(x, dict) else x
TR_RAW, TE_RAW = load_json(get_dataset_file(f"{P6}/Train/train.json")), load_json(get_dataset_file(f"{P6}/Test/test.json"))
IK = next(k for k in TR_RAW[0] if k.lower() in ("id", "doc_id", "case_id")); DK = next(k for k in TR_RAW[0] if k.lower() in ("document", "text", "doc"))
SK = next(k for k in TR_RAW[0] if k.lower() in ("summary", "abstract"))
TR_RAW = [r for r in TR_RAW if doc_text(r[SK]).strip() and doc_text(r[DK]).strip()]
TEST_IDS = [r[IK] for r in TE_RAW]
assert len(set(TEST_IDS)) == len(TEST_IDS)
print(f"train {len(TR_RAW)} documents with a summary | test {len(TE_RAW)} | fields id={IK} document={DK} summary={SK}")
perm = random.Random(0).sample(range(len(TR_RAW)), len(TR_RAW))
DEV_IDX = sorted(perm[:N_DEV]); TRAIN_IDX = sorted(perm[N_DEV:])      # the dev split: fixed for every notebook
if not FULL_RUN:
    TRAIN_IDX = sorted(perm[N_DEV:N_DEV + MAX_TRAIN_DOCS])

def tokenised(name, rows, with_summary):
    cname = f"p6tok_{name}_{MAX_DOC_TOKENS}.pt"
    if not QUICK_TEST:
        try:
            c = torch.load(fetch_model_file(cname), map_location="cpu", weights_only=False)
            if c["ids_order"] == [r[IK] for r in rows]:
                print(f"  {name}: tokens loaded from the bucket cache ({cname})"); return c["docs"]
        except Exception:
            pass
    t0 = time.time(); out = []
    for k, r in enumerate(rows):
        ids, soff = encode_with_sentences(bpe_tokenizer, doc_text(r[DK]), MAX_DOC_TOKENS)
        d = dict(id=r[IK], ids=ids, soff=soff, n=len(ids))
        if with_summary:
            d["sum_ids"] = np.asarray(bpe_tokenizer.encode(doc_text(r[SK])), dtype=np.uint16)
        out.append(d)
        if (k + 1) % 2000 == 0: print(f"    {name}: {k + 1}/{len(rows)} documents tokenised [{time.time() - t0:.0f}s]")
    if not QUICK_TEST:
        torch.save(dict(ids_order=[r[IK] for r in rows], docs=out), f"{WORK}/{cname}"); upload_model_file(f"{WORK}/{cname}", cname)
    return out
t0 = time.time()
assert VOCAB < 65536
ALL_TRAIN = tokenised("train", TR_RAW, True) if not QUICK_TEST else None
if QUICK_TEST:
    keep = DEV_IDX + TRAIN_IDX; tmp = tokenised("train", [TR_RAW[i] for i in keep], True); by = dict(zip(keep, tmp))
    ALL_TRAIN = [by.get(i) for i in range(len(TR_RAW))]
TRAIN_DOCS = [ALL_TRAIN[i] for i in TRAIN_IDX]; DEV_DOCS = [ALL_TRAIN[i] for i in DEV_IDX]
TEST_DOCS = tokenised("test", TE_RAW, False)
DOC_TEXT = {r[IK]: doc_text(r[DK]) for r in TR_RAW + TE_RAW}; REF = {r[IK]: doc_text(r[SK]) for r in TR_RAW}
IDF = compute_idf(TRAIN_DOCS, VOCAB)
st = np.array([len(d["sum_ids"]) for d in TRAIN_DOCS]); dt = np.array([d["n"] for d in TRAIN_DOCS])
print(f"train {len(TRAIN_DOCS)} / dev {len(DEV_DOCS)} / test {len(TEST_DOCS)} documents [{time.time() - t0:.0f}s]; document tokens median {int(np.median(dt))}, p90 {int(np.percentile(dt, 90))}; "
      f"summary tokens p10 {int(np.percentile(st, 10))} median {int(np.median(st))} p90 {int(np.percentile(st, 90))} max {int(st.max())}")
SEP_IDS = [int(x) for x in bpe_tokenizer.encode(BASE["SEP_TEXT"])]
BASE["SEP_IDS"] = SEP_IDS
if BASE["SUM_MAX"] is None:
    BASE["SUM_MAX"] = int(min(np.percentile(st, 90), BASE["CONTEXT"] // 2))
if GEN["MAX_NEW"] is None: GEN["MAX_NEW"] = BASE["SUM_MAX"]
if GEN["MIN_NEW"] is None: GEN["MIN_NEW"] = int(np.percentile(st, 25))
print(f"SUM_MAX {BASE['SUM_MAX']} tokens (longer reference summaries are cut for training); generation MIN_NEW {GEN['MIN_NEW']}, MAX_NEW {GEN['MAX_NEW']}; separator {SEP_IDS}")
SAN_DOCS = {d["id"]: {"ids": d["ids"][:1025].astype(np.int64).tolist()} for d in TRAIN_DOCS[:8]}

print(f"section-aware selection: {set_section_cues(bpe_tokenizer, VOCAB)} cue token ids")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

train 7028 documents with a summary | test 100 | fields id=id document=document summary=summary


  train: tokens loaded from the bucket cache (p6tok_train_32768.pt)


  test: tokens loaded from the bucket cache (p6tok_test_32768.pt)
train 6728 / dev 300 / test 100 documents [3s]; document tokens median 3379, p90 8487; summary tokens p10 316 median 687 p90 1627 max 30260
SUM_MAX 768 tokens (longer reference summaries are cut for training); generation MIN_NEW 454, MAX_NEW 768; separator [4614, 9047, 842, 950, 5804]
section-aware selection: 389 cue token ids


## 9. Load the Phase-1 checkpoint, define how a model is built

In [10]:
t0 = time.time()
_ck = torch.load(fetch_model_file(PHASE1_CHECKPOINT), map_location="cpu", mmap=True)
PHASE1_STATE = _ck["model"] if "model" in _ck else _ck
del _ck
_lm = build_phase1_lm()
PHASE1_PARAMS = sum(p.numel() for p in _lm.parameters())
SANITY_LOSS = gdn_sanity_loss(_lm, SAN_DOCS, device)
print(f"Phase-1 model {MODEL_KEY}: {PHASE1_PARAMS:,} parameters; next-token loss on the documents = {SANITY_LOSS:.3f} nats (random weights would give about {math.log(VOCAB):.1f}) [{time.time()-t0:.0f}s]")
assert SANITY_LOSS < 8.0, "the Phase-1 weights do not seem to be loaded correctly"
del _lm; gc.collect()


def budgets(C):
    """Fill DOC_BUDGET from CONTEXT, SUM_MAX and the separator (decoder-only / LSTM); GDN: encoder budget ENC_MAX_TOK."""
    C = dict(C)
    if KIND == "gdn":
        C["DOC_BUDGET"] = C["DOC_BUDGET"] or (C["ENC_MAX_TOK"] - len(C["SEP_IDS"]))
    else:
        C["DOC_BUDGET"] = C["DOC_BUDGET"] or (C["CONTEXT"] - len(C["SEP_IDS"]) - C["SUM_MAX"] - 1 - (C["PROMPT_LEN"] if C["TECHNIQUE"] == "prompt" else 0))
    assert C["DOC_BUDGET"] > 64, C
    return C


def build_model(C, log=print):
    global DEC_LEN
    if KIND == "gdn":
        DEC_LEN = C["SUM_MAX"] + 2                               # the decoder cache length of the Phase-1 GDN code
        assert C["ENC_MAX_TOK"] + DEC_LEN <= CONTEXT_LEN, "encoder + decoder positions must fit the RoPE table"
    if C.get("RANDOM_INIT"):
        _o = nn.Module.load_state_dict; nn.Module.load_state_dict = lambda self, *a, **k: None
        try: lm = build_phase1_lm()
        finally: nn.Module.load_state_dict = _o
    else:
        lm = build_phase1_lm()
    model = SummModel(KIND, lm, C)
    info = apply_technique(model, C, log)
    if model.lm_head is not None:                               # the decoders' vocabulary projection: trained only with full fine-tuning
        for p in model.lm_head.parameters():
            p.requires_grad = C["TECHNIQUE"] == "full_ft"
    info["params_trainable"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
    info["trainable_pct_of_backbone"] = round(100.0 * info["params_trainable"] / PHASE1_PARAMS, 4)
    return model.to(device), info


def free():
    gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()

Phase-1 model gdn: 144,298,172 parameters; next-token loss on the documents = 3.733 nats (random weights would give about 10.6) [7s]


## 10. Train, summarise dev and test

In [11]:
import hashlib
RES_DIR = f"{WORK}/p6res_{SWEEP_NAME}"
os.makedirs(RES_DIR, exist_ok=True)
SWEEP_START = time.time()


def make_C(ov):
    C = dict(BASE, **ov)
    if "LR" not in ov:
        C["LR"] = LR_BY_TECH.get(C["TECHNIQUE"], C["LR"])
    return budgets(C)


def gen_settings(name):
    return dict(GEN, **GEN_OVERRIDES.get(name, {}))


def cfg_key(C, G, n_train):
    return hashlib.md5(json.dumps([C, G, n_train, len(DEV_DOCS), DEV_GEN, QUICK_TEST, MAX_DOC_TOKENS], sort_keys=True, default=str).encode()).hexdigest()[:10]


def cache_name(setup): return f"p6res_{SWEEP_NAME}__{setup}.pt"
def cache_load(setup, key):
    try:
        r = torch.load(fetch_model_file(cache_name(setup), f"{RES_DIR}/{cache_name(setup)}"), map_location="cpu", weights_only=False)
        return r if r.get("key") == key else None
    except Exception:
        return None
def cache_save(setup, r):
    p = f"{RES_DIR}/{cache_name(setup)}"; torch.save(r, p); upload_model_file(p, cache_name(setup))


def detok(ids):
    return bpe_tokenizer.decode([int(i) for i in ids if int(i) > 3]).strip()


def summarise(model, docs, C, G, log_every=0):
    """Generate a summary for every document; batches of similar prompt length. Returns texts in the order of docs + timing."""
    prompts = [make_prompt(d, C, IDF) for d in docs]
    if KIND == "gdn":
        prompts = [p[-C["ENC_MAX_TOK"]:] for p in prompts]
    order = sorted(range(len(docs)), key=lambda i: len(prompts[i])); out = [None] * len(docs)
    gen = torch.Generator(device=device).manual_seed(C["SEED"]); t0 = time.time(); ntok = 0
    bs = 1 if KIND == "lstm" else G["GEN_BATCH"]
    for a in range(0, len(order), bs):
        idx = order[a:a + bs]
        res = generate(model, [prompts[i] for i in idx], C, G, device, gen)
        for i, ids in zip(idx, res):
            out[i] = detok(ids); ntok += len(ids)
        if log_every and (a // bs) % log_every == 0:
            print(f"    generated {min(a + bs, len(order))}/{len(order)} [{time.time() - t0:.0f}s]")
    secs = time.time() - t0
    return out, dict(seconds=round(secs, 1), docs_per_sec=round(len(docs) / max(secs, 1e-9), 3), new_tokens_per_sec=round(ntok / max(secs, 1e-9), 1))


def kd_targets(C, docs):
    """Sequence-level KD: replace the reference summary by the teacher's generated summary (file from p6_kd_teacher) for the
    documents it covers (a fixed KD_MIX share of them)."""
    t_w = time.time()
    while True:      # the teacher notebook (p6_moe) may still be writing the file: wait for it (up to KD_WAIT_MIN minutes)
        try:
            t = json.load(open(fetch_model_file(C["KD_SUMMARIES"]), encoding="utf-8")); break
        except Exception:
            if QUICK_TEST or (time.time() - t_w) / 60 > KD_WAIT_MIN: raise FileNotFoundError(f"KD summaries {C['KD_SUMMARIES']} not in the bucket (run p6_moe first)")
            print(f"  waiting for the teacher's summaries {C['KD_SUMMARIES']} ({(time.time() - t_w) / 60:.0f} min so far)"); time.sleep(300)
    rng = random.Random(123); out, n = [], 0
    for d in docs:
        if d["id"] in t and t[d["id"]].strip() and rng.random() < C["KD_MIX"]:
            d = dict(d, sum_ids=np.asarray(bpe_tokenizer.encode(t[d["id"]]), dtype=np.uint16)); n += 1
        out.append(d)
    print(f"  KD: {n} of {len(docs)} training documents use the teacher's summary as the target ({C['KD_SUMMARIES']})")
    return out


def wait_for(fname):
    t_w = time.time()
    while True:      # the teacher notebook may still be writing the file: wait for it (up to KD_WAIT_MIN minutes)
        try:
            return fetch_model_file(fname)
        except Exception:
            if QUICK_TEST or (time.time() - t_w) / 60 > KD_WAIT_MIN: raise FileNotFoundError(f"teacher file {fname} not in the bucket (run the teacher notebook first)")
            print(f"  waiting for the teacher file {fname} ({(time.time() - t_w) / 60:.0f} min so far)"); time.sleep(300)


def load_kd_topk(C):
    t = torch.load(wait_for(C["KD_TOPK"]), map_location="cpu", weights_only=False)
    for k in ("DOC_BUDGET", "SELECT", "SUM_MAX", "SEP_IDS", "HEAD_FRAC"):
        assert t["C"][k] == C[k], f"the teacher's top-k file was made with {k}={t['C'][k]}, the student uses {C[k]}: the summary positions would not line up"
    print(f"  word-level KD: teacher top-{t['k']} log-probs for {len(t['docs'])} training documents ({C['KD_TOPK']}), alpha {C['KD_ALPHA']}, temperature {C['KD_TEMPERATURE']}")
    return t["docs"]


def make_kd_topk(wfile, out, k):
    """Teacher job (word-level KD): teacher-forced pass of a trained model over the reference summaries of ALL training documents;
    stores the top-k log-probabilities at every summary position."""
    try:
        fetch_model_file(out); print(f"teacher top-k file {out} already in the bucket"); return
    except Exception:
        pass
    ck = torch.load(fetch_model_file(wfile, f"{RES_DIR}/{wfile}"), map_location="cpu", weights_only=False)
    C = ck["C"]; model, _ = build_model(C); load_trainable_state(model, ck["state"]); model.eval()
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"; col = collate_s2s if KIND == "gdn" else collate_lm
    t0 = time.time(); res = {}
    print(f"== teacher top-{k} log-probs ({wfile}) on {len(TRAIN_DOCS)} training documents")
    with torch.no_grad():
        for i in range(len(TRAIN_DOCS)):
            b = to_device(col(TRAIN_DOCS, [i], C, IDF), device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                lg = model.logits(b)
            m = b["y"] != -100
            lp = torch.log_softmax(lg[m].float(), -1); v, ix = lp.topk(k, -1)
            res[TRAIN_DOCS[i]["id"]] = (ix.cpu().numpy().astype(np.uint16), v.cpu().numpy().astype(np.float16))
            if (i + 1) % 1000 == 0: print(f"    {i + 1}/{len(TRAIN_DOCS)} [{time.time() - t0:.0f}s]")
    path = f"{RES_DIR}/{out}"
    torch.save(dict(k=k, C={x: C[x] for x in ("DOC_BUDGET", "SELECT", "SUM_MAX", "SEP_IDS", "HEAD_FRAC")}, teacher=wfile, docs=res), path)
    print(f"  teacher top-k written: {os.path.getsize(path) / 1e6:.0f} MB, {(time.time() - t0) / 60:.0f} min"); upload_model_file(path, out)
    del model; free()


def run_setup(name, ov, docs=None):
    C = make_C(ov)
    docs = docs or (TRAIN_DOCS[:C["N_TRAIN"]] if C.get("N_TRAIN") else TRAIN_DOCS)      # N_TRAIN: the first N training documents (= the ones the KD teacher summarised)
    G = gen_settings(name); key = cfg_key(C, G, len(docs))
    r = cache_load(name, key)
    if r is not None:
        print(f"[{name}] cached (best epoch {r['best_epoch']}, dev BERTScore F1 {r['dev_scores'].get('bertscore_f1', float('nan')):.4f})"); return r
    print(f"== {name}: {ov}")
    train = kd_targets(C, docs) if C["KD_SUMMARIES"] else docs
    model, info = build_model(C)
    ck_name = f"p6ck_{SWEEP_NAME}__{name}.pt"
    def ck_save(st):
        st["key"] = key; torch.save(st, f"{RES_DIR}/{ck_name}"); ok = upload_model_file(f"{RES_DIR}/{ck_name}", ck_name)
        print(f"  checkpoint after epoch {st['epoch']} " + ("saved to the bucket" if ok is not False else "NOT in the bucket (upload failed)"))
    def ck_load():
        try:
            st = torch.load(fetch_model_file(ck_name), map_location="cpu", weights_only=False); return st if st.get("key") == key else None
        except Exception:
            return None
    if C.get("DAPT_EPOCHS"):
        info.update(dapt(model, docs, DEV_DOCS, C, device))
    kd_topk = load_kd_topk(C) if C.get("KD_TOPK") else None
    res = train_summ(model, train, DEV_DOCS, C, device, IDF, ckpt_save=ck_save if EPOCH_CHECKPOINTS else None, ckpt_load=ck_load if EPOCH_CHECKPOINTS else None, kd_topk=kd_topk)
    del kd_topk
    load_trainable_state(model, res["best_state"])
    if C["SAVE_WEIGHTS"]:
        wname = f"p6w_{SWEEP_NAME}__{name}.pt"
        torch.save(dict(setup=name, overrides=ov, C=C, G=G, kind=KIND, state=res["best_state"], best_epoch=res["best_epoch"]), f"{RES_DIR}/{wname}")
        upload_model_file(f"{RES_DIR}/{wname}", wname)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
    gdocs = DEV_DOCS[:DEV_GEN]
    dev_hyps, dinfo = summarise(model, gdocs, C, G)
    sc = summary_scores(dev_hyps, [REF[d["id"]] for d in gdocs], [DOC_TEXT[d["id"]] for d in gdocs], device)
    try:      # the course evaluator's own code on the dev documents (writes dev_truth.json / <setup>_dev_prediction.json)
        json.dump({d["id"]: REF[d["id"]] for d in gdocs}, open(f"{RES_DIR}/dev_truth.json", "w", encoding="utf-8"), ensure_ascii=False)
        json.dump([dict(id=d["id"], summary=(h.strip() or ".")) for d, h in zip(gdocs, dev_hyps)], open(f"{RES_DIR}/{name}_dev_prediction.json", "w", encoding="utf-8"), ensure_ascii=False)
        sc["official_dev"] = official_eval(f"{RES_DIR}/dev_truth.json", f"{RES_DIR}/{name}_dev_prediction.json")
    except Exception as e:
        print(f"  (official evaluator check skipped: {type(e).__name__}: {e})")
    test_hyps, tinfo = summarise(model, TEST_DOCS, C, G, log_every=10) if (TEST_GEN and FULL_RUN) or QUICK_TEST else ([], {})
    gen_peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    r = dict(key=key, name=name, overrides=ov, C=C, G=G, info=info, history=res["history"], best_epoch=res["best_epoch"], best_dev_loss=res["best_dev_loss"],
             dev_ids=[d["id"] for d in gdocs], dev_hyps=dev_hyps, dev_scores=sc, test_ids=[d["id"] for d in TEST_DOCS], test_hyps=test_hyps,
             train_seconds=res["train_seconds"], peak_mem_gb=res["peak_mem_gb"], gen_peak_mem_gb=gen_peak, train_tokens_per_sec=res["tokens_per_sec"],
             gen_dev=dinfo, gen_test=tinfo, n_train_docs=len(docs), phase1_params=PHASE1_PARAMS)
    cache_save(name, r)
    print(f"### {name}: dev BERTScore F1 {sc['bertscore_f1']:.4f} | ROUGE-1 {sc['rouge1']:.4f} ROUGE-2 {sc['rouge2']:.4f} ROUGE-L {sc['rougeL']:.4f} | {sc['gen_words']:.0f} words (reference {sc['ref_words']:.0f}) | "
          f"repeated 4-grams {sc['repeated_4gram']:.3f} | best epoch {res['best_epoch']} dev loss {res['best_dev_loss']:.4f} | train {res['train_seconds'] / 60:.1f} min, peak {res['peak_mem_gb']} GB | "
          f"generation {dinfo['docs_per_sec']:.2f} docs/s")
    print("    example:", (dev_hyps[0] or "")[:400])
    del model, res; free()
    return r


def make_kd_summaries(wfile, out, n_docs):
    """Teacher job: rebuild a trained model from its saved weights and summarise the first n_docs TRAINING documents (the KD targets)."""
    try:
        fetch_model_file(out); print(f"KD summaries {out} already in the bucket"); return
    except Exception:
        pass
    ck = torch.load(fetch_model_file(wfile, f"{RES_DIR}/{wfile}"), map_location="cpu", weights_only=False)
    C, G = ck["C"], ck["G"]
    model, _ = build_model(C); load_trainable_state(model, ck["state"])
    docs = TRAIN_DOCS[:n_docs]; t0 = time.time(); G = dict(G, GEN_BATCH=max(G["GEN_BATCH"], KD_GEN_BATCH))
    print(f"== KD teacher {wfile}: summarising {len(docs)} training documents")
    hyps, info = summarise(model, docs, C, G, log_every=20)
    path = f"{RES_DIR}/{out}"; json.dump({d["id"]: h for d, h in zip(docs, hyps)}, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    upload_model_file(path, out)
    sc = summary_scores(hyps[:150], [REF[d["id"]] for d in docs[:150]], [DOC_TEXT[d["id"]] for d in docs[:150]], device)
    print(f"  teacher summaries written ({(time.time() - t0) / 60:.0f} min, {info['docs_per_sec']:.2f} docs/s); on 150 of these training documents: BERTScore F1 {sc['bertscore_f1']:.4f}, ROUGE-L {sc['rougeL']:.4f}")
    del model; free()


KD_DONE = set()
def run_kd_jobs():
    """Teacher jobs run as soon as their weight file exists (in this notebook's results or in the bucket)."""
    for _w, _o, _n in KD_JOBS:
        if _o in KD_DONE: continue
        try:
            fetch_model_file(_w, f"{RES_DIR}/{_w}")
        except Exception:
            continue
        make_kd_summaries(_w, _o, _n); KD_DONE.add(_o)
    for _w, _o, _k in KD_TOPK_JOBS:
        if _o in KD_DONE: continue
        try:
            fetch_model_file(_w, f"{RES_DIR}/{_w}")
        except Exception:
            continue
        make_kd_topk(_w, _o, _k); KD_DONE.add(_o)
run_kd_jobs()

RESULTS, SETUP_MIN, FAILED = {}, [], []
for name, ov in SWEEP:
    elapsed = (time.time() - SWEEP_START) / 60; need = 1.15 * max(SETUP_MIN) if SETUP_MIN else 0
    if elapsed + need > MAX_SWEEP_HOURS * 60 and cache_load(name, cfg_key(make_C(ov), gen_settings(name), len(TRAIN_DOCS))) is None:
        print(f"{elapsed:.0f} min used, a setup needs about {need:.0f} min: not starting {name} (Run All again in a new session to continue)"); continue
    t_s = time.time()
    try:
        RESULTS[name] = run_setup(name, ov)
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append(name)
        print(f"!!! setup {name} FAILED ({type(e).__name__}: {str(e)[:200]}); continuing with the next one"); free()
    if time.time() - t_s > 60: SETUP_MIN.append((time.time() - t_s) / 60)
    run_kd_jobs()
missing = [o for _, o, _ in KD_JOBS + KD_TOPK_JOBS if o not in KD_DONE]
if missing: print("!!! KD summaries NOT made (teacher weights missing):", missing)
print(f"sweep done in {(time.time() - SWEEP_START) / 60:.0f} min" + (f"; FAILED setups: {FAILED}" if FAILED else ""))

sweep done in 0 min


## 10b. Decoding study on saved weights (only in the decode notebooks; does nothing when DECODE is empty)

In [12]:
# =====================================================================================================
# Decoding study on SAVED weights (no training): generation length, n-gram blocking, repetition penalty, sampling,
# and inference-time long-document strategies (which part of the judgment the trained model reads; chunk-wise summaries).
# Every job summarises the same DEV_GEN dev documents; the best job(s) also summarise the test set (submission zips).
# =====================================================================================================
DEC_DIR = f"{WORK}/p6_submissions_{SWEEP_NAME}"; os.makedirs(DEC_DIR, exist_ok=True)
DEC_RESULTS, _DEC_MODEL = {}, {}
_f3 = lambda v: f"{v:.3f}" if isinstance(v, (float, int)) else "n/a"


def dec_model(wfile):
    if _DEC_MODEL.get("w") != wfile:
        _DEC_MODEL.clear(); free()
        ck = torch.load(fetch_model_file(wfile, f"{RES_DIR}/{wfile}"), map_location="cpu", weights_only=False)
        m, _ = build_model(ck["C"]); load_trainable_state(m, ck["state"]); m.eval()
        _DEC_MODEL.update(w=wfile, model=m, C=ck["C"], G=ck["G"])
    return _DEC_MODEL["model"], _DEC_MODEL["C"], _DEC_MODEL["G"]


def chunk_prompts(d, C, K, overlap=0.0):
    """Chunk-wise strategy: the judgment is cut into DOC_BUDGET-token chunks (consecutive, or overlapping by `overlap` of a chunk =
    sliding window); at most K of them, evenly spread over the document (always the first and the last)."""
    ids, B, n = d["ids"], C["DOC_BUDGET"], len(d["ids"])
    if n <= B or K <= 1:
        return [make_prompt(d, C, IDF)]
    step = max(1, int(B * (1 - overlap)))
    st = list(range(0, n - B + 1, step))
    if st[-1] != n - B: st.append(n - B)                      # the last window ends at the end of the judgment
    if overlap == 0: st = list(range(0, n, B))              # (non-overlapping: plain consecutive chunks, as in Round 2)
    if len(st) > K:
        st = [st[int(round(j * (len(st) - 1) / (K - 1)))] for j in range(K)]
    sep = np.asarray(C["SEP_IDS"], dtype=np.int64)
    return [np.concatenate([ids[s:s + B].astype(np.int64), sep]) for s in st]


def gen_texts(model, prompts, C, G, seed):
    """Generate for a list of prompts (batched by similar length); returns texts in order + new-token count."""
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i])); out = [None] * len(prompts); ntok = 0
    gen = torch.Generator(device=device).manual_seed(seed)
    bs = 1 if KIND == "lstm" else G["GEN_BATCH"]
    for a in range(0, len(order), bs):
        idx = order[a:a + bs]
        for i, ids in zip(idx, generate(model, [prompts[i] for i in idx], C, G, device, gen)):
            out[i] = detok(ids); ntok += len(ids)
    return out, ntok


def summarise_job(model, docs, C, G, chunks):
    t0 = time.time()
    if not chunks:
        hyps, ntok = gen_texts(model, [make_prompt(d, C, IDF)[-C["ENC_MAX_TOK"]:] if KIND == "gdn" else make_prompt(d, C, IDF) for d in docs], C, G, C["SEED"])
    else:
        sp = chunks if isinstance(chunks, dict) else dict(k=chunks)       # {"k": max chunks, "overlap": 0-0.9, "hier": second pass}
        per = [chunk_prompts(d, C, sp["k"], sp.get("overlap", 0.0)) for d in docs]; parts = {}; ntok = 0
        for k in sorted({len(p) for p in per}):            # documents with the same number of chunks share a length budget
            Gk = dict(G, MAX_NEW=max(16, G["MAX_NEW"] // k), MIN_NEW=G["MIN_NEW"] // k)
            if sp.get("hier"):                              # hierarchical: chunk summaries are only the input of a second pass
                Gk = dict(G, MAX_NEW=max(32, G["MAX_NEW"] // k), MIN_NEW=max(16, G["MIN_NEW"] // (2 * k)))
            items = [(i, j) for i, p in enumerate(per) if len(p) == k for j in range(k)]
            txt, nt = gen_texts(model, [per[i][j] for i, j in items], C, Gk, C["SEED"]); ntok += nt
            parts.update({it: t for it, t in zip(items, txt)})
        hyps = [" ".join(parts[(i, j)] for j in range(len(p))) for i, p in enumerate(per)]
        if sp.get("hier"):        # chunk-level summaries -> document-level summary: the concatenated chunk summaries are summarised again
            multi = [i for i, p in enumerate(per) if len(p) > 1]
            prompts = []
            for i in multi:
                t = np.asarray(bpe_tokenizer.encode(hyps[i]), dtype=np.int64)[:C["DOC_BUDGET"]]
                prompts.append(np.concatenate([t, np.asarray(C["SEP_IDS"], dtype=np.int64)]))
            txt, nt = gen_texts(model, prompts, C, G, C["SEED"]); ntok += nt
            for i, t in zip(multi, txt): hyps[i] = t
    secs = time.time() - t0
    return hyps, dict(seconds=round(secs, 1), docs_per_sec=round(len(docs) / max(secs, 1e-9), 3), new_tokens_per_sec=round(ntok / max(secs, 1e-9), 1))


def run_decode_job(name, wfile, gen_ov, c_ov, chunks):
    model, C0, G0 = dec_model(wfile)
    C = budgets(dict(C0, **c_ov)) if c_ov else C0
    G = dict(G0, **gen_ov)
    key = hashlib.md5(json.dumps([wfile, C, G, chunks, DEV_GEN, QUICK_TEST], sort_keys=True, default=str).encode()).hexdigest()[:10]
    cname = f"p6dec_{SWEEP_NAME}__{name}.pt"
    try:
        r = torch.load(fetch_model_file(cname, f"{RES_DIR}/{cname}"), map_location="cpu", weights_only=False)
        if r.get("key") == key:
            print(f"[{name}] cached: dev BERTScore F1 {r['dev_scores']['bertscore_f1']:.4f}"); return r
    except Exception:
        pass
    gdocs = DEV_DOCS[:DEV_GEN]
    hyps, info = summarise_job(model, gdocs, C, G, chunks)
    sc = summary_scores(hyps, [REF[d["id"]] for d in gdocs], [DOC_TEXT[d["id"]] for d in gdocs], device)
    r = dict(key=key, name=name, weights=wfile, gen_overrides=gen_ov, c_overrides=c_ov, chunks=chunks, C=C, G=G, dev_ids=[d["id"] for d in gdocs], dev_hyps=hyps,
             dev_scores=sc, gen_dev=info, test_hyps=None)
    torch.save(r, f"{RES_DIR}/{cname}"); upload_model_file(f"{RES_DIR}/{cname}", cname)
    print(f"### {name}: dev BERTScore F1 {sc['bertscore_f1']:.4f} | ROUGE-1 {sc['rouge1']:.4f} ROUGE-L {sc['rougeL']:.4f} | {sc['gen_words']:.0f} words | rep4 {sc['repeated_4gram']:.3f} | "
          f"num!doc {_f3(sc['numbers_not_in_doc'])} | {info['docs_per_sec']:.2f} docs/s")
    return r


def test_job(r):
    cname = f"p6dec_{SWEEP_NAME}__{r['name']}.pt"
    if not r.get("test_hyps"):
        model, _, _ = dec_model(r["weights"])
        r["test_hyps"], r["gen_test"] = summarise_job(model, TEST_DOCS, r["C"], r["G"], r["chunks"])
        torch.save(r, f"{RES_DIR}/{cname}"); upload_model_file(f"{RES_DIR}/{cname}", cname)
    jpath, zpath = write_submission([d["id"] for d in TEST_DOCS], r["test_hyps"], DEC_DIR, f"{SWEEP_NAME}_{r['name']}", TEST_IDS)
    check_submission(jpath, TEST_IDS); upload_model_file(zpath)
    print(f"{os.path.basename(zpath):60s} {np.mean([len(h.split()) for h in r['test_hyps']]):.0f} words per summary on test; dev BERTScore F1 {r['dev_scores']['bertscore_f1']:.4f}")


if DECODE:
    t_dec = time.time()
    for name, wfile, gen_ov, c_ov, chunks in DECODE:
        if (time.time() - NB_START) / 3600 > MAX_SWEEP_HOURS:
            print(f"time limit: not starting {name}"); continue
        try:
            DEC_RESULTS[name] = run_decode_job(name, wfile, gen_ov, c_ov, chunks)
        except Exception as e:
            import traceback; traceback.print_exc(); print(f"!!! decode job {name} FAILED: {type(e).__name__}: {str(e)[:200]}"); free()
    rows = sorted(DEC_RESULTS.values(), key=lambda r: -r["dev_scores"]["bertscore_f1"])
    print(f"\nDecoding study on {DEV_GEN} dev documents ({(time.time() - t_dec) / 60:.0f} min). Same trained weights per block; only decoding / input selection change.")
    print(f"{'job':24s} {'weights':34s} {'BERTS-F1':>8s} {'R-1':>6s} {'R-2':>6s} {'R-L':>6s} {'words':>6s} {'rep4':>5s} {'novel3':>6s} {'num!doc':>7s} {'refnum':>6s} {'doc/s':>6s}")
    for r in rows:
        s = r["dev_scores"]
        print(f"{r['name']:24s} {r['weights'][4:38]:34s} {s['bertscore_f1']:8.4f} {s['rouge1']:6.4f} {s['rouge2']:6.4f} {s['rougeL']:6.4f} {s['gen_words']:6.0f} {s['repeated_4gram']:5.3f} {s['novel_3gram']:6.3f} "
              f"{_f3(s['numbers_not_in_doc']):>7s} {_f3(s['ref_numbers_covered']):>6s} {r['gen_dev']['docs_per_sec']:6.2f}")
    # test files: the best TEST_TOP jobs by dev BERTScore (+ any listed in TEST_ALSO), unless the best one is the setting the training notebook already submitted
    pick = [r for r in rows if r["name"] not in DEC_BASELINES][:TEST_TOP] + [DEC_RESULTS[n] for n in TEST_ALSO if n in DEC_RESULTS]
    for r in pick:
        if (time.time() - NB_START) / 3600 > MAX_SWEEP_HOURS + 0.7:
            print(f"time limit: no test file for {r['name']}"); continue
        test_job(r)
    json.dump({n: dict(weights=r["weights"], gen_overrides=r["gen_overrides"], c_overrides=r["c_overrides"], chunks=r["chunks"], dev_scores=r["dev_scores"], gen_dev=r["gen_dev"])
               for n, r in DEC_RESULTS.items()}, open(f"{WORK}/decode_{SWEEP_NAME}.json", "w"), indent=1, default=str)
    upload_model_file(f"{WORK}/decode_{SWEEP_NAME}.json")

Technique lorma: trainable 1,376,256 of 145,674,428 parameters (0.9538% of the Phase-1 model's 144,298,172); stored size 583.7 MB


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (86 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded decode_p6_decode_gdn.json to bucket kavincg23/cs781-models


## 11. Comparison table

In [13]:
rows = sorted(RESULTS.items(), key=lambda kv: -kv[1]["dev_scores"].get("bertscore_f1", 0))
print(f"Dev scores on {DEV_GEN} held-out training documents (never trained on). BERTScore = {BERTSCORE_MODEL} F1 (the primary metric). words = average generated length (reference length in brackets).")
print(f"{'setup':26s} {'BERTS-F1':>8s} {'R-1':>6s} {'R-2':>6s} {'R-L':>6s} {'words':>11s} {'rep4':>5s} {'novel3':>6s} {'num!doc':>7s} {'refnum':>6s} {'devloss':>7s} {'ep':>3s} {'train M':>8s} {'%bb':>6s} {'GB':>5s} {'min':>6s} {'gen doc/s':>9s}")
for n, r in rows:
    s, i = r["dev_scores"], r["info"]
    f = lambda v: f"{v:.3f}" if isinstance(v, float) else "  n/a"
    print(f"{n:26s} {s.get('bertscore_f1', float('nan')):8.4f} {s['rouge1']:6.4f} {s['rouge2']:6.4f} {s['rougeL']:6.4f} {s['gen_words']:5.0f} ({s['ref_words']:4.0f}) {s['repeated_4gram']:5.3f} {s['novel_3gram']:6.3f} {f(s['numbers_not_in_doc']):>7s} {f(s['ref_numbers_covered']):>6s} "
          f"{r['best_dev_loss']:7.4f} {r['best_epoch']:3d} {i['params_trainable'] / 1e6:8.2f} {i['trainable_pct_of_backbone']:6.2f} {(r['peak_mem_gb'] or 0):5.1f} {r['train_seconds'] / 60:6.1f} {r['gen_dev']['docs_per_sec']:9.3f}")
print("rep4 = share of repeated 4-grams inside a summary; novel3 = share of 3-grams not in the document; num!doc = share of the numbers in the summary that are not in the document (a hallucination proxy); refnum = share of the reference's numbers that the summary contains.")

Dev scores on 100 held-out training documents (never trained on). BERTScore = roberta-large F1 (the primary metric). words = average generated length (reference length in brackets).
setup                      BERTS-F1    R-1    R-2    R-L       words  rep4 novel3 num!doc refnum devloss  ep  train M    %bb    GB    min gen doc/s
rep4 = share of repeated 4-grams inside a summary; novel3 = share of 3-grams not in the document; num!doc = share of the numbers in the summary that are not in the document (a hallucination proxy); refnum = share of the reference's numbers that the summary contains.


## 12. Submission files

In [14]:
SUB_DIR = f"{WORK}/p6_submissions_{SWEEP_NAME}"
os.makedirs(SUB_DIR, exist_ok=True)
for name, r in sorted(RESULTS.items(), key=lambda kv: -kv[1]["dev_scores"].get("bertscore_f1", 0)):
    if not r["test_hyps"] and not QUICK_TEST:
        continue
    jpath, zpath = write_submission(r["test_ids"], r["test_hyps"], SUB_DIR, f"{SWEEP_NAME}_{name}", TEST_IDS)
    check_submission(jpath, TEST_IDS)
    print(f"{os.path.basename(zpath):60s} {np.mean([len(h.split()) for h in r['test_hyps']]):.0f} words per summary on test; dev BERTScore F1 {r['dev_scores'].get('bertscore_f1', float('nan')):.4f}; file check passed")
    upload_model_file(zpath)

## 13. Summary

In [15]:
SUMMARY = dict(sweep=SWEEP_NAME, model=MODEL_KEY, kind=KIND, phase1_checkpoint=PHASE1_CHECKPOINT, phase1_params=PHASE1_PARAMS, sanity_loss=SANITY_LOSS, base=BASE, gen=GEN,
               results={n: {k: v for k, v in r.items() if k not in ("dev_hyps", "test_hyps", "test_ids")} for n, r in RESULTS.items()}, total_minutes=round((time.time() - NB_START) / 60, 1))
path = f"{WORK}/summary_{SWEEP_NAME}.json"
json.dump(SUMMARY, open(path, "w"), indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
upload_model_file(path)
for p in sorted(os.listdir(SUB_DIR)):
    print("  ", f"{SUB_DIR}/{p}", os.path.getsize(f"{SUB_DIR}/{p}") // 1024, "KB")
if UPLOAD_FAILED:
    print("!" * 80 + f"\n{len(UPLOAD_FAILED)} FILES COULD NOT BE UPLOADED: " + ", ".join(UPLOAD_FAILED) + "\n" + "!" * 80)
print(f"done in {(time.time() - NB_START) / 60:.0f} min.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded summary_p6_decode_gdn.json to bucket kavincg23/cs781-models
   /kaggle/working/p6_submissions_p6_decode_gdn/p6_decode_gdn_lorma_reppen1.2_norep3_prediction.json 352 KB
   /kaggle/working/p6_submissions_p6_decode_gdn/p6_decode_gdn_lorma_reppen1.2_prediction.json 373 KB
   /kaggle/working/p6_submissions_p6_decode_gdn/submission_p6_decode_gdn_lorma_reppen1.2.zip 77 KB
   /kaggle/working/p6_submissions_p6_decode_gdn/submission_p6_decode_gdn_lorma_reppen1.2_norep3.zip 77 KB
done in 26 min.
